# ModernBERT by hand

**Part 2 of "LLMs the Hard Way"** (branch `modernbert-by-hand`). The decoder notebooks on `master` built GPT-2 and SmolLM2 by hand, ending in a KV-cache decode step. This notebook does the same for an **encoder**: [answerdotai/ModernBERT-large](https://huggingface.co/answerdotai/ModernBERT-large), loaded from its real checkpoint as a plain dict of tensors, with no library model.

We implement and inspect every piece ourselves, then check the result against the Hugging Face implementation:

- tokenization and token embeddings, and why position information is needed;
- RoPE, with ModernBERT's two bases (global and local layers);
- Q, K, V, multi-head attention, and **bidirectional** attention (no causal mask), compared with SmolLM2's causal attention;
- local (sliding-window) vs global attention layers;
- LayerNorm, residual connections and the GeGLU MLP;
- one complete encoder layer, then the full 28-layer stack;
- the output `[batch, sequence, hidden]`: token representations and mean pooling;
- **a classification head on top of the encoder.**

**The end result:** our forward pass matches the reference `last_hidden_state` to about 3e-5, and the checkpoint's own masked-LM head fills in masked words correctly.

**The idea this notebook ends on.** An encoder doesn't generate text. It turns the input into vectors, one per token. Pool them into one vector, put a small **head** on top (here a linear layer), and the encoder becomes a model that **decides**: a class, a yes / no, a score. The encoder does the language work; the head only learns to read the decision off the vector. That's how "BERT for classification" works, and the rest of this series builds on it.

Whether it *works* is a separate question, and it depends on the job:
- **A fixed set of labels** (sentiment, spam, a fixed intent taxonomy, language ID): a fixed head with one output per class is the classic, proven setup.
- **Our job, routing a conversation to agents that change** (agents are added, merged and re-scoped): Phase 10 shows a fixed head can't do it. It never reads the agents' descriptions, and every new agent needs a new output and retraining.

So we keep the idea and change the head. `02_frozen_router.ipynb` (branch `frozen-router`) puts the agent's description **into the input** and has the head output **one score per (conversation, agent) pair**. It's still a head on a pooled encoder vector, and it can now score agents it has never seen. The cell numbers there continue from this notebook (Cell 34 onward).

**Runs on:** a CPU is enough (the comparisons in Phases 2–9 run on CPU on purpose, for exact numbers). On Kaggle or Colab, run it top to bottom.

Each unit is **one Markdown cell + one Code cell**.

In [ ]:
# Cell 1 — Environment sanity check

import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device :", "CUDA" if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
!pip install -q "transformers==5.17.0"

---
# Phase 1 — Establish what ModernBERT is

## Cell 2 — Encoder vs decoder

**What.** Compare two Transformer families on a 4-token sequence, `the cat sat down`:

```text
SmolLM2 (decoder):    tokens → next token
ModernBERT (encoder): tokens → contextual representations
```

**Why.** Everything later depends on this difference. SmolLM2 *generates*: at position `i` it predicts token `i+1`, so it must never see the future and uses a **causal mask**. ModernBERT *understands*: it produces one vector per token that summarizes that token **in the context of the whole sequence**, left and right. It has no "next token" to predict, so it doesn't need a causal mask and doesn't generate text autoregressively.

**Math.** Attention lets query token `i` read from key token `j` only where the visibility mask allows it:

- decoder: `visible[i, j] = (j ≤ i)`, a lower-triangular matrix with `S(S+1)/2` true entries
- encoder: `visible[i, j] = True` for all `i, j`, giving `S²` true entries

The training objectives differ in the same way. A decoder learns `P(token_{i+1} | tokens_≤i)`. A BERT-style encoder learns **masked language modelling**: hide a token with `[MASK]` and recover it from *both* sides.

**Shapes.** Each visibility mask is `[S, S]` = `[4, 4]`. Row = query token, column = key token.

**Expect.** In the decoder table, `the` sees only itself and `down` sees everything. In the encoder table every row is full. The masked-LM example shows `cat` being recovered from `the … sat down`, which needs right-hand context.

In [ ]:
# Cell 2 — Encoder vs decoder: which tokens can each token see?

tokens = ["the", "cat", "sat", "down"]
S = len(tokens)

# Decoder (SmolLM2): query i may attend to keys 0..i  -> lower-triangular
decoder_visibility = torch.tril(torch.ones(S, S, dtype=torch.bool))

# Encoder (ModernBERT): query i may attend to every key -> all True
encoder_visibility = torch.ones(S, S, dtype=torch.bool)

print("decoder_visibility:", tuple(decoder_visibility.shape), "| encoder_visibility:", tuple(encoder_visibility.shape))
print()


def show_visibility(title, visibility):
    print(title)
    print("query \\ key".ljust(13) + "".join(t.ljust(6) for t in tokens))
    for i, t in enumerate(tokens):
        row = "".join(("✓" if visibility[i, j] else "·").ljust(6) for j in range(S))
        print(t.ljust(13) + row)
    print()


show_visibility("Decoder (causal) — SmolLM2", decoder_visibility)
show_visibility("Encoder (bidirectional) — ModernBERT", encoder_visibility)

print("What each token's output can depend on:")
for i, t in enumerate(tokens):
    dec = [tokens[j] for j in range(S) if decoder_visibility[i, j]]
    enc = [tokens[j] for j in range(S) if encoder_visibility[i, j]]
    print(f"  {t:5s} | decoder: {str(dec):32s} | encoder: {enc}")

print("\nDecoder objective — predict the NEXT token:")
for i in range(S - 1):
    print(f"  {tokens[:i + 1]} -> {tokens[i + 1]!r}")

print("\nEncoder objective (masked LM) — recover a hidden token from BOTH sides:")
print(f"  {['the', '[MASK]', 'sat', 'down']} -> position 1 = 'cat'")

assert decoder_visibility.sum().item() == S * (S + 1) // 2
assert encoder_visibility.sum().item() == S * S

## Cell 3 — Model configuration, from the real checkpoint

**What.** Read the configuration of the real pretrained ModernBERT checkpoint named by `CHECKPOINT` into our own config object, and load its pretrained weights as a **plain dictionary of tensors**.

**Why.** We use the actual pretrained model as the source of truth, but we compute the forward pass ourselves. So we need exactly two things from the checkpoint:

```text
ModernBERT checkpoint
    ├── config.json        → every dimension of the model
    └── model.safetensors  → every pretrained weight tensor
```

`hf_hub_download` and `safetensors` only fetch and read files. No library model is built here, and no library code runs a forward pass.

**The fields.**

| Field | Meaning | ModernBERT-large (used) | ModernBERT-base |
|---|---|---|---|
| `vocab_size` (V) | rows in the embedding table | 50,368 | 50,368 |
| `hidden_size` (H) | width of every token vector | 1,024 | 768 |
| `num_layers` (L) | stacked encoder blocks | 28 | 22 |
| `num_attention_heads` | heads per layer, `head_dim = H / heads` | 16 × 64 | 12 × 64 |
| `intermediate_size` (I) | MLP inner width | 2,624 | 1,152 |
| `max_position_embeddings` | longest supported sequence | 8,192 | 8,192 |
| `local_attention_window` | sliding-window width for local layers | 128 (±64 tokens) | 128 (±64 tokens) |
| `global_attn_every_n_layers` | every n-th layer attends globally | 3 | 3 |
| `global_rope_theta` / `local_rope_theta` | RoPE bases for global / local layers | 160,000 / 10,000 | 160,000 / 10,000 |

**Math.** The parameter count follows from the shapes (there are no biases anywhere):

```text
embeddings   V·H  (token table)  +  H  (embedding norm)
per layer    3H·H (Wqkv)  +  H·H (Wo)  +  H·2I (MLP Wi)  +  I·H (MLP Wo)  +  2H (two norms)
             − H for layer 0, which has no attention norm (the embedding norm already precedes it)
final norm   H
```

**Shapes.** Each tensor is listed with its shape. `Wqkv` is `[3H, H] = [3072, 1024]` (Q, K and V stacked), and the MLP input `Wi` is `[2I, H] = [5248, 1024]` because of its gated activation (Cell 24).

**Expect.** The formula matches the checkpoint exactly: **394,781,696** encoder parameters. The checkpoint also contains a masked-LM prediction head (`head.*`, `decoder.bias`), which we set aside because routing doesn't use it. Layers 0, 3, 6, …, 27 (10 of 28) are global-attention layers, and the other 18 are local.

> **Memory.** The weights take about 1.6 GB in float32. Later cells also load a reference model for comparison (another ~1.6 GB). If the kernel dies, close other notebook kernels to free RAM.

**Switching checkpoints.** `CHECKPOINT` selects the model, and the default is now **ModernBERT-large**. Every code cell reads its sizes from `config`, so the same hand-written forward pass runs either checkpoint. The shapes and numbers quoted in these notes are **ModernBERT-large's** (H = 1,024, 28 layers: 10 global and 18 local, 16 heads of 64, I = 2,624). For **base**, read H = 768, 22 layers (8 global, 14 local), 12 heads and I = 1,152. The local window (128) and both RoPE bases are the same.

In [ ]:
# Cell 3 — Model configuration + pretrained weights, read from the real checkpoint

import json
from dataclasses import dataclass

from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

torch.manual_seed(42)

# Which pretrained checkpoint to load. Both share the architecture built in this notebook:
#   "answerdotai/ModernBERT-base"  : 22 layers, H = 768,  12 heads, I = 1152  (149M parameters)
#   "answerdotai/ModernBERT-large" : 28 layers, H = 1024, 16 heads, I = 2624  (395M parameters)
CHECKPOINT = "answerdotai/ModernBERT-large"

config_path = hf_hub_download(CHECKPOINT, "config.json")
weights_path = hf_hub_download(CHECKPOINT, "model.safetensors")

with open(config_path) as f:
    raw_config = json.load(f)


@dataclass
class ModernBertConfig:
    vocab_size: int
    hidden_size: int
    num_layers: int
    num_attention_heads: int
    intermediate_size: int
    max_position_embeddings: int
    local_attention_window: int
    global_attn_every_n_layers: int
    global_rope_theta: float
    local_rope_theta: float
    norm_eps: float
    pad_token_id: int
    cls_token_id: int
    sep_token_id: int

    @property
    def head_dim(self):
        return self.hidden_size // self.num_attention_heads

    def is_global_layer(self, layer_idx):
        return layer_idx % self.global_attn_every_n_layers == 0


config = ModernBertConfig(
    vocab_size=raw_config["vocab_size"],
    hidden_size=raw_config["hidden_size"],
    num_layers=raw_config["num_hidden_layers"],
    num_attention_heads=raw_config["num_attention_heads"],
    intermediate_size=raw_config["intermediate_size"],
    max_position_embeddings=raw_config["max_position_embeddings"],
    local_attention_window=raw_config["local_attention"],
    global_attn_every_n_layers=raw_config["global_attn_every_n_layers"],
    global_rope_theta=raw_config["global_rope_theta"],
    local_rope_theta=raw_config["local_rope_theta"],
    norm_eps=raw_config["norm_eps"],
    pad_token_id=raw_config["pad_token_id"],
    cls_token_id=raw_config["cls_token_id"],
    sep_token_id=raw_config["sep_token_id"],
)

assert config.hidden_size % config.num_attention_heads == 0, "H must split evenly across heads"
assert config.head_dim % 2 == 0, "RoPE rotates pairs of dimensions"

print(f"Checkpoint: {CHECKPOINT}\n")
for field in ["vocab_size", "hidden_size", "num_layers", "num_attention_heads", "head_dim",
              "intermediate_size", "max_position_embeddings", "local_attention_window",
              "global_attn_every_n_layers", "global_rope_theta", "local_rope_theta", "norm_eps"]:
    print(f"  {field:28s} {getattr(config, field)}")

global_layers = [i for i in range(config.num_layers) if config.is_global_layer(i)]
print(f"\n  global-attention layers: {global_layers}")
print(f"  local-attention layers : {[i for i in range(config.num_layers) if i not in global_layers]}")

# ---- Pretrained weights as a plain dict of tensors (no library model) ----
checkpoint_tensors = load_file(weights_path)
weights = {name.removeprefix("model."): t for name, t in checkpoint_tensors.items() if name.startswith("model.")}
mlm_head = sorted(name for name in checkpoint_tensors if not name.startswith("model."))
del checkpoint_tensors

for t in weights.values():
    t.requires_grad_(False)

print(f"\nLoaded {len(weights)} encoder tensors; set aside the masked-LM head: {mlm_head}\n")
for name, t in weights.items():
    if ".layers." not in "." + name or name.startswith(("layers.0.", "layers.1.")):
        print(f"  {name:36s} {str(tuple(t.shape)):14s} {t.dtype}")
print(f"  ... layers.2 – layers.{config.num_layers - 1} repeat the layers.1 pattern")

H, I, V, L = config.hidden_size, config.intermediate_size, config.vocab_size, config.num_layers
expected = (V * H + H) + L * (3 * H * H + H * H + H * 2 * I + I * H + 2 * H) - H + H
actual = sum(t.numel() for t in weights.values())
print(f"\nEncoder parameters: {actual:,} (formula: {expected:,})")

assert actual == expected
assert "layers.0.attn_norm.weight" not in weights, "layer 0 has no attention norm"
assert weights["layers.0.attn.Wqkv.weight"].shape == (3 * H, H)
assert weights["layers.0.mlp.Wi.weight"].shape == (2 * I, H)

---
# Phase 2 — Input representation

## Cell 4 — Token IDs

**What.** Use ModernBERT's own tokenizer to turn text into token IDs:

```text
text → tokens → token IDs
```

**Why.** The encoder only understands integers, and each integer is a row number in the pretrained embedding table (Cell 5). Because we load real weights, we must use the exact tokenizer the model was trained with. A home-made vocabulary would map words to the wrong rows. The tokenizer is a preprocessing utility, not part of the forward pass, so using the library here is fine.

ModernBERT uses a **byte-level BPE** vocabulary of 50,368 entries. Common words are single tokens, and rarer words split into several subword pieces. `Ġ` marks a token that starts with a space, so `the` (sentence start) and `Ġthe` (mid-sentence) are **different tokens**.

**Special tokens.**

| Token | ID | Role |
|---|---|---|
| `[CLS]` | 50281 | placed first; a conventional "whole sequence" slot (Cell 31 discusses whether to use it) |
| `[SEP]` | 50282 | ends a segment; later separates conversation turns from the agent description |
| `[PAD]` | 50283 | filler so batched sequences have equal length; excluded via the attention mask |
| `[MASK]` | 50284 | hides a token during masked-language-model pretraining |
| `[UNK]` | 50280 | unknown token (rarely needed with byte-level BPE) |

**Math.** `encode(text) = [CLS] + BPE(text) + [SEP]`, then right-pad with `[PAD]` to the longest sequence in the batch. The **attention mask** is `1` for real tokens and `0` for padding.

**Shapes.** Two sentences padded together give `input_ids: [B, S]` and `attention_mask: [B, S]`, with `B = 2`.

**Expect.** Common words map to single tokens, and the shorter sentence ends in `[PAD]` (50283) with mask 0 at those positions.

In [ ]:
# Cell 4 — Token IDs with ModernBERT's own tokenizer: text -> tokens -> token IDs

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

special = ["[CLS]", "[SEP]", "[PAD]", "[MASK]", "[UNK]"]
special_ids = {t: tokenizer.convert_tokens_to_ids(t) for t in special}
print("Vocabulary size:", len(tokenizer))
print("Special token IDs:", special_ids)

assert len(tokenizer) == config.vocab_size
assert special_ids["[CLS]"] == config.cls_token_id
assert special_ids["[SEP]"] == config.sep_token_id
assert special_ids["[PAD]"] == config.pad_token_id

CLS_ID, SEP_ID, PAD_ID, MASK_ID = (special_ids[t] for t in ["[CLS]", "[SEP]", "[PAD]", "[MASK]"])

texts = ["The dog bites the man.", "My card payment failed yesterday, please help!"]

for text in texts:
    ids = tokenizer(text)["input_ids"]
    print(f"\ntext   : {text!r}")
    print(f"tokens : {tokenizer.convert_ids_to_tokens(ids)}")
    print(f"ids    : {ids}")

batch = tokenizer(texts, padding=True, return_tensors="pt")
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

print("\ninput_ids:", tuple(input_ids.shape), "= [B, S]")
print(input_ids)
print("attention_mask:", tuple(attention_mask.shape))
print(attention_mask)

assert input_ids.shape == attention_mask.shape
assert (input_ids[:, 0] == CLS_ID).all(), "every sequence starts with [CLS]"
assert ((input_ids == PAD_ID) == (attention_mask == 0)).all(), "mask is 0 exactly where [PAD] is"
for b in range(input_ids.shape[0]):
    last_real = attention_mask[b].sum().item() - 1
    assert input_ids[b, last_real] == SEP_ID, "every sequence ends with [SEP] before padding"

## Cell 5 — Token embeddings

**What.** Turn each token ID into its **pretrained** vector by looking it up in ModernBERT's embedding table:

```text
input_ids  [B, S]        integers
    ↓  row lookup in W_emb [V, H]
X          [B, S, H]     one 1,024-dim vector per token
```

**Why.** IDs are arbitrary labels: ID 4370 isn't "bigger" than ID 637. The embedding table holds a learned point in H-dimensional space for every token. In the real checkpoint, those points already encode a great deal of meaning from pretraining. Attention then refines each vector using its context.

**Math.** An embedding is a table lookup, `X[b, s] = W_emb[input_ids[b, s]]`. This equals `one_hot(id) @ W_emb` without materialising the one-hot. In PyTorch it is plain indexing, `W_emb[input_ids]`, which is exactly what we do. We don't use `nn.Embedding`, because the weights come from the checkpoint and the lookup is just indexing.

**The dimensions.**

- `B` (batch): independent sequences processed together, here 2
- `S` (sequence): token positions, padded to the longest sequence in the batch
- `H` (hidden): features per token, 1,024 for ModernBERT-large (768 for base)

**ModernBERT details.**
- There is **no position-embedding table**. Unlike the original BERT, position is injected later, inside attention, with RoPE (Phase 3).
- The real model normalizes this output (`embeddings.norm`) before layer 0. We implement that with the other normalizations in Cell 23.
- The `[PAD]` row is **not** zero in this checkpoint. Padding is made harmless by the attention mask, not by the embedding values.

**Validation.** We load the reference ModernBERT implementation **only for comparison** and check that its embedding output equals ours exactly. It never feeds into our computation.

**Expect.** `X` is `[2, S, 1024]`, and the maximum difference from the reference is exactly `0.0`.

In [ ]:
# Cell 5 — Token embeddings: [B, S] -> [B, S, H] using the pretrained table

W_emb = weights["embeddings.tok_embeddings.weight"]      # [V, H]

X = W_emb[input_ids]                                     # our lookup: plain row indexing

B, S = input_ids.shape
H = config.hidden_size

print("embedding table W_emb:", tuple(W_emb.shape), "= [V, H]")
print("input_ids            :", tuple(input_ids.shape), "= [B, S]")
print("X                    :", tuple(X.shape), "= [B, S, H]")

assert X.shape == (B, S, H)
assert torch.equal(X[0, 0], W_emb[CLS_ID]), "[CLS] vector = row CLS_ID of the table"

print("\nFirst 6 dims of each token in sequence 0:")
for s in range(S):
    token = tokenizer.convert_ids_to_tokens(input_ids[0, s].item())
    print(f"  pos {s:2d} {token:10s} {X[0, s, :6]}")

print(f"\n[PAD] row norm: {W_emb[PAD_ID].norm():.3f}  (not zero: padding is handled by the attention mask)")

# ---- Validation against the reference implementation (comparison only) ----
from transformers import ModernBertModel

reference = ModernBertModel.from_pretrained(CHECKPOINT, attn_implementation="eager").eval()

with torch.no_grad():
    X_ref = reference.embeddings.tok_embeddings(input_ids)

max_diff = (X - X_ref).abs().max().item()
print(f"\nours vs reference token embeddings: max |difference| = {max_diff}")
assert torch.equal(X, X_ref)

## Cell 6 — Why position information is needed

**What.** Compare `the dog bites the man` with `the man bites the dog`. Both tokenize into **exactly the same five BPE tokens**, only in a different order, and they mean very different things.

**Why.** Embeddings alone carry no order: `Ġdog` is the same row of the table wherever it appears. Self-attention without position information is also blind to order. It compares tokens using only their *contents*, so shuffling the input just shuffles the output. This property is called **permutation equivariance**. Without position information, even the real pretrained model couldn't tell who bit whom.

**Math.** For self-attention `Attn(E) = softmax(E W_qᵀ (E W_kᵀ)ᵀ / √d) · E W_vᵀ`, permuting the input rows by a permutation matrix `P` gives

`Attn(P E) = P · Attn(E)`

So each token's output depends on *which* tokens are present, never on *where* they are, and any order-independent summary (such as the mean over tokens) is identical.

**Shapes.** We use the **real** layer-0 projections for attention head 0, with no RoPE (a preview of Phase 4). `Wqkv` `[3072, 1024]` splits into `W_q, W_k, W_v`, each `[1024, 1024]`. Head 0 is the first 64 rows of each, `[64, 1024]`. The sentence `[1, 7]` → embeddings `[1, 7, 1024]` → head-0 output `[1, 7, 64]`. We skip the embedding normalization here: it acts on each token separately, so it doesn't change the argument.

**Expect.**
- Same sorted token IDs, different order.
- `Ġdog`'s and `Ġman`'s outputs are **identical** across the two sentences, up to float32 rounding (about 1e-8).
- The mean-pooled sentence vectors are identical too. This is the problem RoPE solves next.

In [ ]:
# Cell 6 — Without position information, "the dog bites the man" == "the man bites the dog"

ids_a = tokenizer("the dog bites the man")["input_ids"]
ids_b = tokenizer("the man bites the dog")["input_ids"]

print("A:", tokenizer.convert_ids_to_tokens(ids_a), ids_a)
print("B:", tokenizer.convert_ids_to_tokens(ids_b), ids_b)

assert sorted(ids_a) == sorted(ids_b), "same multiset of tokens"
assert ids_a != ids_b, "different order"

E_a = W_emb[torch.tensor([ids_a])]                       # [1, S, H]
E_b = W_emb[torch.tensor([ids_b])]

# Real layer-0 projections, head 0 only, and NO RoPE (preview of Phase 4)
W_q, W_k, W_v = weights["layers.0.attn.Wqkv.weight"].chunk(3, dim=0)   # each [H, H]
d = config.head_dim
W_q0, W_k0, W_v0 = W_q[:d], W_k[:d], W_v[:d]                           # each [d, H]


def attention_without_position(E):
    Q, K, Vv = E @ W_q0.T, E @ W_k0.T, E @ W_v0.T          # [1, S, d] each
    scores = Q @ K.transpose(-2, -1) / d ** 0.5             # [1, S, S]
    return torch.softmax(scores, dim=-1) @ Vv               # [1, S, d]


out_a = attention_without_position(E_a)
out_b = attention_without_position(E_b)
print("\nE_a:", tuple(E_a.shape), "| head-0 output:", tuple(out_a.shape))

print("\nSame token, different position — is the output different?")
for token in ["Ġdog", "Ġman", "Ġbites"]:
    tid = tokenizer.convert_tokens_to_ids(token)
    pa, pb = ids_a.index(tid), ids_b.index(tid)
    diff = (out_a[0, pa] - out_b[0, pb]).abs().max().item()
    print(f"  {token:7s}: position {pa} in A, {pb} in B -> max |difference| = {diff:.2e}")
    assert torch.allclose(out_a[0, pa], out_b[0, pb], atol=1e-5)

sentence_a, sentence_b = out_a.mean(dim=1), out_b.mean(dim=1)
print(f"\nMean-pooled sentence vectors differ by {(sentence_a - sentence_b).abs().max().item():.2e}")
assert torch.allclose(sentence_a, sentence_b, atol=1e-5)

print("\nConclusion: without position information the model cannot tell who bit whom.")

---
# Phase 3 — RoPE

## Cell 7 — RoPE motivation

**What.** Compare the two ways of giving a Transformer position information:

```text
absolute (original BERT):  x_pos = x + p[pos]        → added to the token vector before layer 0
rotary   (ModernBERT):     Q, K → rotate by an angle that depends on the position
```

**Why.** Cell 6 showed that attention without position can't tell `dog bites man` from `man bites dog`.

- **Absolute embeddings** (original BERT) add a learned vector `p[pos]` to each token. Position becomes mixed into *every* feature, including the values V. The table also has a fixed size (512 in BERT), and relative distances have to be inferred indirectly.
- **RoPE** leaves the token vector alone and **rotates the query and key** just before the attention dot product, by an angle proportional to the position. A rotation doesn't change a vector's length, V is never touched, and the score between two tokens depends only on **how far apart** they are.

**Math.** Take one 2-D pair of a query `q` and key `k`, using RoPE pair 0 (dims 0 and 32, rotating 1 radian per token):

```text
absolute:  score(m, n) = (q + p_m) · (k + p_n)      depends on m and n separately
RoPE:      score(m, n) = (R(m)q) · (R(n)k) = qᵀ R(n − m) k      depends only on n − m
```

**Shapes.** Each `q`/`k` here is a 2-vector taken from the real layer-0 head-0 projections of `Ġdog` (query) and `Ġman` (key). The full implementation works on `[B, heads, S, head_dim]` (Cell 9).

**Expect.** For position pairs that are all 2 apart, the **absolute** scores vary with where the pair sits, while the **RoPE** scores are identical. Adding a position vector changes the length of `q`, and rotating it doesn't.

In [ ]:
# Cell 7 — Absolute position embeddings vs RoPE, on one real 2-D pair

import math

dog_id = tokenizer.convert_tokens_to_ids("Ġdog")
man_id = tokenizer.convert_tokens_to_ids("Ġman")

q_full = W_emb[dog_id] @ W_q0.T      # [64]  real layer-0 head-0 query of Ġdog
k_full = W_emb[man_id] @ W_k0.T      # [64]  real layer-0 head-0 key of Ġman

half = d // 2
q2 = torch.stack([q_full[0], q_full[half]])   # RoPE pair 0 = dims (0, 32)
k2 = torch.stack([k_full[0], k_full[half]])
print("q pair:", q2, "| k pair:", k2)


def absolute_position_vector(pos, scale):
    """A sinusoidal absolute position vector (original-Transformer style), scaled to the pair's size."""
    return scale * torch.tensor([math.sin(pos), math.cos(pos)])


def rotation(angle):
    c, s = math.cos(angle), math.sin(angle)
    return torch.tensor([[c, -s], [s, c]])


scale = q2.norm().item()
pairs_2_apart = [(0, 2), (3, 5), (10, 12), (50, 52)]

print(f"\n{'q pos':>5} {'k pos':>5} | {'absolute score':>15} | {'RoPE score':>11}")
print("-" * 45)
rope_scores = []
for m, n in pairs_2_apart:
    abs_score = (q2 + absolute_position_vector(m, scale)) @ (k2 + absolute_position_vector(n, scale))
    rope_score = (rotation(m * 1.0) @ q2) @ (rotation(n * 1.0) @ k2)     # pair 0 turns 1 rad per token
    rope_scores.append(rope_score)
    print(f"{m:>5} {n:>5} | {abs_score.item():>15.5f} | {rope_score.item():>11.5f}")

assert all(torch.allclose(s, rope_scores[0], atol=1e-5) for s in rope_scores), "RoPE: offset-only"

print("\nLength of q at position 3:")
print(f"  original              : {q2.norm():.5f}")
print(f"  absolute (q + p_3)    : {(q2 + absolute_position_vector(3, scale)).norm():.5f}")
print(f"  RoPE (R(3) q)         : {(rotation(3.0) @ q2).norm():.5f}")

## Cell 8 — RoPE mathematics

**What.** Compute RoPE's rotation **frequencies** for ModernBERT's two layer types, and check the properties of the 2-D rotation that RoPE applies to each pair of dimensions.

**Why.** A 64-dim head is treated as **32 independent 2-D planes**. Each plane rotates at its own speed, its frequency. Fast planes separate nearby positions, and slow planes change only over long distances. ModernBERT uses **two bases** because its layers see different distances:

- **global** layers attend over the whole sequence (up to 8,192 tokens), so they use `θ = 160,000` (slower rotations)
- **local** layers only attend within ±64 tokens, so they use `θ = 10,000`

**Math.** Frequency of pair `i` (`d = head_dim = 64`, `i = 0 … 31`):

```text
inv_freq[i] = 1 / θ^(2i / d)            wavelength[i] = 2π / inv_freq[i] tokens per full turn
```

At position `p`, pair `i` is rotated by the angle `p · inv_freq[i]`:

```text
[x']   [cos θ  -sin θ] [x]
[y'] = [sin θ   cos θ] [y]
```

Properties we verify: `RᵀR = I` (lengths preserved), `det R = 1` (a pure rotation), `R(a)R(b) = R(a+b)` (angles add), and therefore `R(m)ᵀR(n) = R(n − m)`, which is the relative-position property.

**Which dimensions form a pair.** ModernBERT pairs dimension `i` with `i + 32` (the "rotate-half" convention), **not** adjacent dimensions `(2i, 2i+1)`.

**Shapes.** `inv_freq` is `[32]` for each layer type.

**Expect.**
- Pair 0 turns 1 rad per token for both bases (a wavelength of about 6.3 tokens).
- Pair 31 is about 15× slower for global layers than for local ones (a full turn every ~690,000 tokens vs ~47,000).
- Our frequencies match the reference model's buffers.
- All rotation identities hold.

In [ ]:
# Cell 8 — RoPE frequencies and the 2-D rotation

def rope_inv_freq(head_dim, theta):
    i2 = torch.arange(0, head_dim, 2, dtype=torch.float32)      # 0, 2, ..., head_dim-2
    return 1.0 / (theta ** (i2 / head_dim))                     # [head_dim / 2]


inv_freq_global = rope_inv_freq(config.head_dim, config.global_rope_theta)
inv_freq_local = rope_inv_freq(config.head_dim, config.local_rope_theta)

print("inv_freq_global:", tuple(inv_freq_global.shape), "| inv_freq_local:", tuple(inv_freq_local.shape))
print(f"\n{'pair':>4} {'dims':>9} | {'global freq':>12} {'wavelength':>12} | {'local freq':>11} {'wavelength':>11}")
print("-" * 70)
for i in [0, 1, 4, 8, 16, 24, 31]:
    fg, fl = inv_freq_global[i].item(), inv_freq_local[i].item()
    print(f"{i:>4} {f'({i},{i + half})':>9} | {fg:>12.6f} {2 * math.pi / fg:>12,.0f} | {fl:>11.6f} {2 * math.pi / fl:>11,.0f}")

# Validation: our frequencies vs the reference model's buffers
assert torch.allclose(inv_freq_global, reference.rotary_emb.full_attention_inv_freq)
assert torch.allclose(inv_freq_local, reference.rotary_emb.sliding_attention_inv_freq)
print("\nMatches reference full_attention / sliding_attention frequencies.")

# The 2-D rotation and its properties
a, b, m, n = 0.7, 1.9, 3.0, 8.0
I2 = torch.eye(2)
assert torch.allclose(rotation(a).T @ rotation(a), I2, atol=1e-6), "R^T R = I"
assert abs(torch.det(rotation(a)).item() - 1.0) < 1e-6, "det R = 1"
assert torch.allclose(rotation(a) @ rotation(b), rotation(a + b), atol=1e-6), "angles add"
assert torch.allclose(rotation(m).T @ rotation(n), rotation(n - m), atol=1e-6), "relative angle"

print("\nRotate (1, 0) by 90°:", rotation(math.pi / 2) @ torch.tensor([1.0, 0.0]))
print("R^T R = I, det R = 1, R(a)R(b) = R(a+b), R(m)^T R(n) = R(n-m): all verified")
print(f"\nRoPE pairs for head_dim={config.head_dim}: (0,{half}), (1,{half + 1}), ..., ({half - 1},{config.head_dim - 1})")

## Cell 9 — RoPE implementation

**What.** Implement RoPE for full attention tensors, apply it to the **real** layer-0 queries and keys of `the dog bites the man`, and check the result against the reference implementation.

**Why.** This is the exact operation every ModernBERT attention layer performs on Q and K before computing scores. Writing it ourselves shows that RoPE is nothing more than elementwise multiplies and one half-swap.

**Math.** For every position `p`, build the angle vector `angles[p] = p · inv_freq`, which is `[32]`. Duplicate it to `[64]` so it lines up with both halves of the head:

```text
cos[p] = cos([angles, angles])      sin[p] = sin([angles, angles])
rotate_half([x₁, x₂]) = [−x₂, x₁]   (x₁ = dims 0–31, x₂ = dims 32–63)
RoPE(x) = x · cos + rotate_half(x) · sin
```

For pair `i` this is exactly the 2-D rotation from Cell 8: `x₁ᵢ' = x₁ᵢ cos − x₂ᵢ sin` and `x₂ᵢ' = x₁ᵢ sin + x₂ᵢ cos`.

**Shapes.**

```text
E          [1, 7, 1024]          raw token embeddings
qkv        [1, 7, 3072]         E @ Wqkvᵀ
Q, K       [1, 16, 7, 64]       = [B, heads, S, head_dim]  (head split previewed here, explained in Cells 11–12)
cos, sin   [7, 64]              broadcast over B and heads
RoPE(Q)    [1, 16, 7, 64]       same shape as Q
```

Layer 0 is a **global** layer, so it uses `θ = 160,000`. We feed raw embeddings here, since the embedding normalization comes in Cell 23. The comparison is still exact because both implementations get the same Q and K.

**Expect.**
- Position 0 is unchanged (angle 0), and the vector lengths are unchanged.
- The pair-0 value of `Ġdog` matches a hand-applied 2×2 rotation.
- Our `cos`/`sin` and rotated Q/K match the reference for both global and local layers, with a maximum difference of about 0.

In [ ]:
# Cell 9 — RoPE implementation: [B, heads, S, head_dim] -> same shape

def rope_cos_sin(seq_len, inv_freq):
    positions = torch.arange(seq_len, dtype=torch.float32, device=inv_freq.device)   # [S]
    angles = positions[:, None] * inv_freq[None, :]               # [S, d/2]
    angles = torch.cat([angles, angles], dim=-1)                  # [S, d]  same angle for dims i and i+d/2
    return angles.cos(), angles.sin()


def rotate_half(x):
    h = x.shape[-1] // 2
    return torch.cat([-x[..., h:], x[..., :h]], dim=-1)           # [x1, x2] -> [-x2, x1]


def apply_rope(x, cos, sin):
    """x: [..., S, d]; cos/sin: [S, d] (broadcast over batch and heads)."""
    return x * cos + rotate_half(x) * sin


# Real layer-0 Q and K for "the dog bites the man"
E = W_emb[torch.tensor([ids_a])]                                  # [1, 7, 1024]
qkv = E @ weights["layers.0.attn.Wqkv.weight"].T                  # [1, 7, 3072]
B_, S_, _ = qkv.shape
q, k, v = qkv.view(B_, S_, 3, config.num_attention_heads, d).unbind(dim=2)   # each [1, 7, 12, 64]
Q, K = q.transpose(1, 2), k.transpose(1, 2)                                    # [1, 12, 7, 64]

cos_g, sin_g = rope_cos_sin(S_, inv_freq_global)                  # layer 0 is a global layer
Q_rot, K_rot = apply_rope(Q, cos_g, sin_g), apply_rope(K, cos_g, sin_g)

print("E:", tuple(E.shape), "| qkv:", tuple(qkv.shape), "| Q:", tuple(Q.shape), "| cos:", tuple(cos_g.shape), "| RoPE(Q):", tuple(Q_rot.shape))
assert Q_rot.shape == Q.shape == (1, config.num_attention_heads, S_, d)

# Small inspectable example: head 0, token Ġdog, RoPE pair 0 = dims (0, 32)
p_dog = ids_a.index(dog_id)
before = torch.stack([Q[0, 0, p_dog, 0], Q[0, 0, p_dog, half]])
after = torch.stack([Q_rot[0, 0, p_dog, 0], Q_rot[0, 0, p_dog, half]])
by_hand = rotation(p_dog * inv_freq_global[0].item()) @ before
print(f"\nĠdog at position {p_dog}, head 0, pair 0: before {before} -> after {after} (2x2 by hand: {by_hand})")
assert torch.allclose(after, by_hand, atol=1e-5)

assert torch.allclose(Q_rot[:, :, 0], Q[:, :, 0]), "position 0: angle 0, unchanged"
assert torch.allclose(Q_rot.norm(dim=-1), Q.norm(dim=-1), atol=1e-4), "rotation preserves length"
print("Position 0 unchanged; every vector's length unchanged.")

# Validation against the reference implementation
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb as reference_apply_rope

position_ids = torch.arange(S_)[None]                             # [1, S]
for layer_type, inv_freq in [("full_attention", inv_freq_global), ("sliding_attention", inv_freq_local)]:
    cos_ours, sin_ours = rope_cos_sin(S_, inv_freq)
    cos_ref, sin_ref = reference.rotary_emb(E, position_ids, layer_type)      # [1, S, d]
    Q_ref, K_ref = reference_apply_rope(Q, K, cos_ref, sin_ref)
    Q_ours, K_ours = apply_rope(Q, cos_ours, sin_ours), apply_rope(K, cos_ours, sin_ours)
    diff_cs = max((cos_ours - cos_ref[0]).abs().max().item(), (sin_ours - sin_ref[0]).abs().max().item())
    diff_qk = max((Q_ours - Q_ref).abs().max().item(), (K_ours - K_ref).abs().max().item())
    print(f"{layer_type:17s}: cos/sin max |diff| = {diff_cs:.1e} | rotated Q/K max |diff| = {diff_qk:.1e}")
    assert diff_cs < 1e-6 and diff_qk < 1e-5

## Cell 10 — Verify that position changes Q and K

**What.** Put the **same** vector at different positions and show that RoPE gives it different representations. Then return to Cell 6's example and show that RoPE now separates `the dog bites the man` from `the man bites the dog`.

**Why.** This closes the loop on Cell 6. Before RoPE, identical tokens produced identical attention outputs wherever they stood. After RoPE, the query and key carry their position, so the attention pattern, and with it the output, depends on word order.

**Math.**

- Same vector, different position: `RoPE(q, p) = R(p) q`. Its length stays `‖q‖`, but its direction changes with `p`, so `cos∠(R(0)q, R(p)q)` drops below 1.
- Relative property: `R(m)q · R(n)k = qᵀ R(n−m) k`. Every query/key pair with the same offset `n − m` gets the same score, wherever it sits.

**Shapes.** The single head-0 vectors are `[64]`. The dog/man attention reuses Cell 6's head-0 attention, now with RoPE applied to Q and K, giving `[1, 7, 64]` outputs.

**Expect.**
- The rotated `Ġdog` query keeps its length at every position, while its cosine similarity to the position-0 version varies.
- Scores for query/key position pairs `(2,4)`, `(3,5)`, `(10,12)` and `(100,102)` are identical, and other offsets give different scores.
- With RoPE, `Ġdog`'s attention output now **differs** between the two sentences, by far more than float rounding.

> **Conclusion:** RoPE injects position-dependent information into Q/K, so attention can distinguish token positions and relative relationships.

In [ ]:
# Cell 10 — Same vector, different positions: RoPE makes them different

q_dog = Q[0, 0, p_dog]                                     # [64] head-0 query of Ġdog (before RoPE)
k_man = K[0, 0, ids_a.index(man_id)]                       # [64] head-0 key of Ġman (before RoPE)

cos_long, sin_long = rope_cos_sin(200, inv_freq_global)    # angles for positions 0..199


def rope_at(vec, pos):
    return apply_rope(vec, cos_long[pos], sin_long[pos])


print("Same Ġdog query vector placed at different positions:")
print(f"{'pos':>4} | {'length':>8} | {'cosine sim. to pos 0':>20}")
for pos in [0, 1, 2, 5, 10, 50, 100]:
    rotated = rope_at(q_dog, pos)
    cos_sim = torch.nn.functional.cosine_similarity(rotated, q_dog, dim=0).item()
    print(f"{pos:>4} | {rotated.norm():>8.4f} | {cos_sim:>20.4f}")
    assert torch.allclose(rotated.norm(), q_dog.norm(), atol=1e-4)

print("\nScore depends only on the offset n - m:")
same_offset = [(2, 4), (3, 5), (10, 12), (100, 102)]
scores = [rope_at(q_dog, m) @ rope_at(k_man, n) for m, n in same_offset]
for (m, n), s in zip(same_offset, scores):
    print(f"  q at {m:>3}, k at {n:>3} (offset {n - m:+d}): {s.item():.5f}")
assert all(torch.allclose(s, scores[0], atol=1e-3) for s in scores)
for m, n in [(2, 2), (2, 7), (4, 2)]:
    print(f"  q at {m:>3}, k at {n:>3} (offset {n - m:+d}): {(rope_at(q_dog, m) @ rope_at(k_man, n)).item():.5f}")

# Back to Cell 6: the same head-0 attention, now WITH RoPE on Q and K
def attention_with_rope(E):
    Qh, Kh, Vh = E @ W_q0.T, E @ W_k0.T, E @ W_v0.T             # [1, S, d]
    cos_s, sin_s = rope_cos_sin(E.shape[1], inv_freq_global)
    Qh, Kh = apply_rope(Qh, cos_s, sin_s), apply_rope(Kh, cos_s, sin_s)
    scores = Qh @ Kh.transpose(-2, -1) / d ** 0.5               # [1, S, S]
    return torch.softmax(scores, dim=-1) @ Vh                   # [1, S, d]


out_a_rope = attention_with_rope(E_a)
out_b_rope = attention_with_rope(E_b)

print("\nthe dog bites the man  vs  the man bites the dog — head-0 output, with RoPE:")
for token in ["Ġdog", "Ġman"]:
    tid = tokenizer.convert_tokens_to_ids(token)
    pa, pb = ids_a.index(tid), ids_b.index(tid)
    without = (out_a[0, pa] - out_b[0, pb]).abs().max().item()
    with_rope = (out_a_rope[0, pa] - out_b_rope[0, pb]).abs().max().item()
    print(f"  {token:5s}: max |difference| without RoPE = {without:.1e}  |  with RoPE = {with_rope:.1e}")
    assert with_rope > 1e-4 and with_rope > 1000 * without     # RoPE makes the difference, without it ~0

print("\nRoPE injects position-dependent information into Q/K, so attention can tell the two sentences apart.")

---
# Phase 4 — Self-attention

## Cell 11 — Q, K, V

**What.** Project the real layer-0 input into **queries, keys and values** with the pretrained `Wqkv` weights, for the sentence `My card payment failed, so I need a new card.`

**Why.** Attention lets every token gather information from other tokens. Each token plays three roles:

- **Query (Q):** what this token is looking for
- **Key (K):** what this token offers to be matched against
- **Value (V):** the information this token hands over when it is attended to

**The layer-0 input.** ModernBERT normalizes the token embeddings before layer 0 (`embeddings.norm`, a LayerNorm with a scale but no bias). We apply it here so that our Q, K and V are exactly the model's. Cell 23 explains LayerNorm in detail.

```text
x = LayerNorm(W_emb[input_ids])          [B, S, H]
```

**Math.** The checkpoint stores the three projections **stacked** in one matrix, `Wqkv` `[3H, H]`. PyTorch linear weights are stored as `[out, in]`, so `X @ Wq` from the spec is `x @ W_qᵀ` here:

```text
W_q, W_k, W_v = split(Wqkv, 3)            each [H, H] = [1024, 1024]
Q = x @ W_qᵀ    K = x @ W_kᵀ    V = x @ W_vᵀ
```

There are no biases, because ModernBERT removes them from all linear layers.

**Shapes.**

```text
input_ids     [1, S]
x             [1, S, 1024]
Q, K, V       [1, S, 1024]     B = 1 sentence, S tokens, H = 1024 features
```

**Expect.** All three projections are `[1, S, 1024]`. Our layer-0 input matches the reference `embeddings` output, and our concatenated `[Q, K, V]` matches the reference `Wqkv` output.

In [ ]:
# Cell 11 — Q, K, V from the real layer-0 Wqkv

text = "My card payment failed, so I need a new card."
input_ids_1 = tokenizer(text, return_tensors="pt")["input_ids"]          # [1, S]
tokens_1 = tokenizer.convert_ids_to_tokens(input_ids_1[0])
B, S = input_ids_1.shape
H = config.hidden_size
print("tokens:", tokens_1)


def layer_norm(x, weight, eps=config.norm_eps):
    """ModernBERT LayerNorm: normalize each token vector, then scale (no bias). Detailed in Cell 23."""
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    return (x - mean) / torch.sqrt(var + eps) * weight


x = layer_norm(W_emb[input_ids_1], weights["embeddings.norm.weight"])   # layer-0 input [B, S, H]

W_q, W_k, W_v = weights["layers.0.attn.Wqkv.weight"].chunk(3, dim=0)   # each [H, H]

Q_proj = x @ W_q.T                                                     # [B, S, H]
K_proj = x @ W_k.T
V_proj = x @ W_v.T

print("\ninput_ids:", tuple(input_ids_1.shape), "| x:", tuple(x.shape))
print("Wqkv:", tuple(weights["layers.0.attn.Wqkv.weight"].shape), "-> W_q, W_k, W_v:", tuple(W_q.shape))
print("Q:", tuple(Q_proj.shape), "| K:", tuple(K_proj.shape), "| V:", tuple(V_proj.shape))
assert Q_proj.shape == K_proj.shape == V_proj.shape == (B, S, H)

# Validation against the reference implementation
with torch.no_grad():
    x_ref = reference.embeddings(input_ids_1)                          # embeddings + norm
    qkv_ref = reference.layers[0].attn.Wqkv(x_ref)                     # layer 0 has no attn_norm

diff_x = (x - x_ref).abs().max().item()
diff_qkv = (torch.cat([Q_proj, K_proj, V_proj], dim=-1) - qkv_ref).abs().max().item()
print(f"\nlayer-0 input vs reference : max |diff| = {diff_x:.1e}")
print(f"[Q, K, V] vs reference Wqkv: max |diff| = {diff_qkv:.1e}")
assert diff_x < 1e-5 and diff_qkv < 1e-4

## Cell 12 — Multi-head splitting

**What.** Split the 1,024 features of Q, K and V into **16 heads of 64 features each**, then apply RoPE to every Q and K head (layer 0 is a global layer, so `θ = 160,000`):

```text
[B, S, H]  →  view  →  [B, S, heads, head_dim]  →  transpose  →  [B, heads, S, head_dim]
```

**Why.** A single attention pattern per layer would be limiting. With 16 heads, each head runs its **own** attention in its own 64-dim subspace, so different heads can track different relationships, such as a word's neighbours, repeated words, or a noun and its verb. The transpose puts `heads` next to `B`, so that the score computation in Cell 13 runs for all heads in one batched matmul.

**Math.**

```text
H = heads × head_dim        1024 = 16 × 64
head h uses features h·64 … h·64 + 63 of Q, K and V
```

No numbers change. `view` and `transpose` only reinterpret the same values, which is why merging the heads back (Cell 17) loses nothing. RoPE then rotates each head's Q and K exactly as in Cell 9.

**Shapes.**

```text
Q_proj               [1, S, 1024]
→ view               [1, S, 16, 64]
→ transpose(1, 2)    [1, 16, S, 64]  = Q_heads   (same for K_heads, V_heads)
RoPE(Q/K_heads)      [1, 16, S, 64]
```

**Expect.**
- Head 5 of token 3 is exactly features 320–383 of that token's projection.
- Undoing the transpose and view recovers the original tensor exactly.
- RoPE leaves each head vector's length unchanged.

In [ ]:
# Cell 12 — Multi-head split: [B, S, H] -> [B, heads, S, head_dim], then RoPE on Q and K

n_heads, d = config.num_attention_heads, config.head_dim
assert H == n_heads * d


def split_heads(t):
    B_, S_, _ = t.shape
    return t.view(B_, S_, n_heads, d).transpose(1, 2)        # [B, heads, S, head_dim]


Q_heads, K_heads, V_heads = split_heads(Q_proj), split_heads(K_proj), split_heads(V_proj)

print("Q_proj :", tuple(Q_proj.shape), "= [B, S, H]")
print("view   :", tuple(Q_proj.view(B, S, n_heads, d).shape), "= [B, S, heads, head_dim]")
print("Q_heads:", tuple(Q_heads.shape), "= [B, heads, S, head_dim]")
print(f"H = heads × head_dim: {H} = {n_heads} × {d}")

# Head h of token s is just a 64-wide slice of the original feature vector
h, s = 5, 3
assert torch.equal(Q_heads[0, h, s], Q_proj[0, s, h * d:(h + 1) * d])
print(f"\nhead {h}, token {s} ({tokens_1[s]!r}) == Q_proj features {h * d}..{(h + 1) * d - 1}")

# Splitting is lossless: undo it exactly
assert torch.equal(Q_heads.transpose(1, 2).reshape(B, S, H), Q_proj)
print("transpose + reshape back recovers Q_proj exactly")

# RoPE on every head's Q and K (layer 0 is global)
cos_0, sin_0 = rope_cos_sin(S, inv_freq_global)
Q_heads_rope, K_heads_rope = apply_rope(Q_heads, cos_0, sin_0), apply_rope(K_heads, cos_0, sin_0)

print("\nafter RoPE:", tuple(Q_heads_rope.shape), "(V is never rotated:", tuple(V_heads.shape), ")")
assert torch.allclose(Q_heads_rope.norm(dim=-1), Q_heads.norm(dim=-1), atol=1e-4)
assert torch.allclose(K_heads_rope.norm(dim=-1), K_heads.norm(dim=-1), atol=1e-4)

## Cell 13 — Attention scores

**What.** Compare every query with every key, in every head:

```python
scores = Q @ K.transpose(-2, -1) / sqrt(head_dim)
```

**Why.** The score says how well token `i`'s query matches token `j`'s key. Softmax (Cell 15) turns each row of scores into weights, which decide how much of each token's value flows into token `i`'s new representation.

**Math.** For head `h`:

```text
scores[b, h, i, j] = (q_i · k_j) / √64        q_i = RoPE'd query of token i, k_j = RoPE'd key of token j
```

**What `[i, j]` means.** Row `i` is the **query** token (the one being updated). Column `j` is the **key** token (the one being looked at). `scores[0, h, i, j]` is how strongly token `i` wants to read token `j` in head `h`. The matrix is **not symmetric**, because `q_i · k_j ≠ q_j · k_i` in general.

Dividing by `√head_dim = 8` keeps the scores' spread roughly independent of `head_dim`. Without it, dot products grow with dimension, and softmax would collapse onto a single key.

**Shapes.**

```text
Q_heads_rope              [1, 12, S, 64]
K_heads_rope.transpose    [1, 12, 64, S]
scores                    [1, 12, S, S]       one S × S matrix per head
```

**Expect.**
- `scores` is `[1, 12, S, S]` with no masking yet. Every token scores every other token, including tokens to its right.
- A hand-computed dot product reproduces one entry.
- The matrix isn't symmetric.
- In head 0, the `[CLS]` and `[SEP]` **columns** score highest for almost every query. Many heads use the special tokens as a default place to put attention (an "attention sink") when no other token is especially relevant.

In [ ]:
# Cell 13 — Attention scores: [B, heads, S, S]

scores = Q_heads_rope @ K_heads_rope.transpose(-2, -1) / d ** 0.5      # [1, 12, S, S]

print("Q:", tuple(Q_heads_rope.shape), "@ K^T:", tuple(K_heads_rope.transpose(-2, -1).shape), "->", tuple(scores.shape))
assert scores.shape == (B, n_heads, S, S)

# One entry by hand: head 0, query token i, key token j
h, i, j = 0, 2, 10
manual = (Q_heads_rope[0, h, i] @ K_heads_rope[0, h, j]) / d ** 0.5
print(f"\nscores[0, {h}, {i}, {j}] = {scores[0, h, i, j]:.4f}  (by hand: {manual:.4f})  "
      f"— how much {tokens_1[i]!r} (query) matches {tokens_1[j]!r} (key)")
assert torch.allclose(manual, scores[0, h, i, j], atol=1e-5)

assert not torch.allclose(scores[0, h], scores[0, h].T), "scores are not symmetric"
print(f"not symmetric: scores[{i},{j}] = {scores[0, h, i, j]:.3f} vs scores[{j},{i}] = {scores[0, h, j, i]:.3f}")

labels = [t.replace("Ġ", "_")[:7] for t in tokens_1]
print(f"\nHead {h} scores (rows = query token i, columns = key token j):")
print(" " * 9 + "".join(f"{l:>8}" for l in labels))
for r, l in enumerate(labels):
    print(f"{l:>9}" + "".join(f"{v:8.2f}" for v in scores[0, h, r].tolist()))

---
# Phase 5 — The critical ModernBERT difference

## Cell 14 — No causal mask

**What.** Compare the attention pattern of head 0 on our sentence under two rules:

```text
SmolLM2 (decoder):    token i → tokens ≤ i      (causal mask)
ModernBERT (encoder): token i → every token     (no causal mask)
```

**Why.** An encoder's job is to understand each token **in its full context**. The second `card` in `…so I need a new card.` and the first `card` in `My card payment failed…` can only be told apart, and understood, if each sees the words on **both** sides. A causal mask would throw half of that context away. It's only needed when the model must not see the future, as in next-token generation, and an encoder doesn't generate.

**Math.** A causal mask sets every score above the diagonal to `−∞` before softmax, so `softmax` gives those positions exactly 0:

```text
causal:         scores[i, j] = −∞  for j > i
bidirectional:  scores[i, j] unchanged for all j
```

**Shapes.** One head's score matrix is `[S, S] = [14, 14]`. Both masks are boolean `[14, 14]`.

**Expect.**
- The causal heatmap is lower-triangular (white above the diagonal).
- The bidirectional heatmap is filled everywhere.
- The first `Ġcard` (position 2) puts a large share of its attention on tokens to its **right** under bidirectional attention, and exactly 0 under the causal mask.

In [ ]:
# Cell 14 — Bidirectional (ModernBERT) vs causal (SmolLM2) attention, head 0 of layer 0

import matplotlib.pyplot as plt

causal_mask = torch.tril(torch.ones(S, S, dtype=torch.bool))     # j <= i
bidirectional_mask = torch.ones(S, S, dtype=torch.bool)          # every j

head0 = scores[0, 0]                                             # [S, S]
causal_probs = torch.softmax(head0.masked_fill(~causal_mask, float("-inf")), dim=-1)
bidir_probs = torch.softmax(head0.masked_fill(~bidirectional_mask, float("-inf")), dim=-1)

print("scores (one head):", tuple(head0.shape), "| masks:", tuple(causal_mask.shape))
assert (causal_probs[~causal_mask] == 0).all(), "causal: no attention to the future"
assert (bidir_probs > 0).all(), "bidirectional: every token sees every token"

i = 2   # the first Ġcard
print(f"\nQuery {tokens_1[i]!r} (position {i}) — share of attention on tokens to its RIGHT:")
print(f"  causal       : {causal_probs[i, i + 1:].sum():.3f}")
print(f"  bidirectional: {bidir_probs[i, i + 1:].sum():.3f}")

labels = [t.replace("Ġ", "_") for t in tokens_1]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, probs, title in [(axes[0], causal_probs, "SmolLM2-style causal: token i → tokens ≤ i"),
                         (axes[1], bidir_probs, "ModernBERT bidirectional: token i → every token")]:
    ax.imshow(probs.numpy(), cmap="Blues", vmin=0, vmax=probs.max().item())
    ax.set_xticks(range(S), labels, rotation=90)
    ax.set_yticks(range(S), labels)
    ax.set_xlabel("key token j")
    ax.set_ylabel("query token i")
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Cell 15 — Softmax attention

**What.** Turn every row of scores into a probability distribution, for all 16 heads of layer 0:

```python
attention = torch.softmax(scores, dim=-1)
```

**Why.** Scores are unbounded similarities. Softmax converts each query's row into **non-negative weights that sum to 1**: how token `i` divides its attention among all tokens `j`. These weights are applied to the values in Cell 16.

**Math.**

```text
attention[b, h, i, j] = exp(scores[b, h, i, j]) / Σ_j' exp(scores[b, h, i, j'])
Σ_j attention[b, h, i, j] = 1      for every b, h, i
```

The softmax runs over the **last** dimension (`dim=-1`, the keys), so each row is normalized independently.

**Shapes.** `scores` `[1, 16, S, S]` → `attention` `[1, 16, S, S]`.

**Validation.** We run the reference implementation's layer-0 attention module on the same input. It returns its attention weights, which should match ours.

**Expect.**
- Every one of the `1 × 16 × 14` rows sums to 1 (within about 1e-6).
- Our weights match the reference's to about 1e-6.
- The `Ġcard` row shows where that token gathers information from.

In [ ]:
# Cell 15 — Softmax over keys: [B, heads, S, S], every row sums to 1

attention = torch.softmax(scores, dim=-1)                          # [1, 12, S, S]
row_sums = attention.sum(dim=-1)                                   # [1, 12, S]

print("attention:", tuple(attention.shape), "| row sums:", tuple(row_sums.shape))
print(f"row sums: min {row_sums.min():.7f}  max {row_sums.max():.7f}")
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5)

i = 11  # the second Ġcard ("a new card")
top = torch.topk(attention[0, 0, i], k=4)
print(f"\nHead 0, query {tokens_1[i]!r} (position {i}) attends most to:")
for w, j in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {tokens_1[j]!r:12s} position {j:2d}  weight {w:.3f}")

# Validation: the reference layer-0 attention module, same input, returns its attention weights
position_ids_1 = torch.arange(S)[None]
cos_ref0, sin_ref0 = reference.rotary_emb(x_ref, position_ids_1, "full_attention")
with torch.no_grad():
    ref_attn_output, ref_attention = reference.layers[0].attn(x_ref, position_embeddings=(cos_ref0, sin_ref0), attention_mask=None)

diff = (attention - ref_attention).abs().max().item()
print(f"\nours vs reference attention weights: max |diff| = {diff:.1e}")
assert diff < 1e-5

## Cell 16 — Weighted values

**What.** Each token's new representation (per head) is the **attention-weighted average of the value vectors**:

```python
context = attention @ V
```

**Why.** Scores and softmax decide *where to look*. The values decide *what is copied*. A token that attends 30% to `Ġpayment` receives 30% of `Ġpayment`'s value vector in that head. This is the step where information actually moves between tokens.

**Math.** For head `h` and query token `i`:

```text
context[b, h, i] = Σ_j attention[b, h, i, j] · V[b, h, j]
```

**Shapes.**

```text
attention   [B, heads, S, S]          [1, 12, 14, 14]
      ×
V_heads     [B, heads, S, head_dim]   [1, 12, 14, 64]
      ↓
context     [B, heads, S, head_dim]   [1, 12, 14, 64]
```

The `S × S` weights mix the `S` value rows, so the result has one 64-dim vector per token per head.

**Expect.** `context` is `[1, 12, 14, 64]`, and an explicit weighted sum over the 14 value vectors reproduces one entry exactly.

In [ ]:
# Cell 16 — Weighted values: [B, heads, S, S] @ [B, heads, S, d] -> [B, heads, S, d]

context = attention @ V_heads                                      # [1, 12, S, 64]

print("attention:", tuple(attention.shape), "@ V:", tuple(V_heads.shape), "->", tuple(context.shape))
assert context.shape == (B, n_heads, S, d)

h, i = 0, 11
weighted_sum = sum(attention[0, h, i, j] * V_heads[0, h, j] for j in range(S))     # explicit Σ_j a_ij v_j
assert torch.allclose(weighted_sum, context[0, h, i], atol=1e-5)
print(f"\nhead {h}, token {tokens_1[i]!r}: context = Σ_j attention[i, j] · V[j]  (checked by explicit sum)")

contrib = attention[0, h, i, :, None] * V_heads[0, h]              # [S, d] each token's contribution
shares = contrib.norm(dim=-1) / contrib.norm(dim=-1).sum()
print("largest contributions to that context vector:")
for j in torch.argsort(shares, descending=True)[:4].tolist():
    print(f"  {tokens_1[j]!r:12s} weight {attention[0, h, i, j]:.3f}  share of contribution {shares[j]:.3f}")

## Cell 17 — Merge heads

**What.** Undo Cell 12's split and lay the 16 head outputs side by side again:

```text
[B, heads, S, head_dim]  →  transpose(1, 2)  →  [B, S, heads, head_dim]  →  reshape  →  [B, S, H]
```

**Why.** The rest of the layer (output projection, residual, MLP) works on one `H = 1,024` vector per token. Merging concatenates the 16 heads' 64-dim results into that vector: features `0–63` come from head 0, `64–127` from head 1, and so on.

**Math.** `merged[b, s, h·64 : (h+1)·64] = context[b, h, s, :]`. It's the exact inverse of the split, so no values change.

**Shapes.** `context` `[1, 16, 14, 64]` → `[1, 14, 16, 64]` → `merged` `[1, 14, 1024]`.

**Expect.** `merged` is `[1, 14, 1024]`, and slicing head 3's block out of it gives back `context[0, 3]` exactly.

In [ ]:
# Cell 17 — Merge heads: [B, heads, S, d] -> [B, S, H]

merged = context.transpose(1, 2).reshape(B, S, H)                  # [1, S, 1024]

print("context:", tuple(context.shape), "-> transpose:", tuple(context.transpose(1, 2).shape), "-> merged:", tuple(merged.shape))
assert merged.shape == (B, S, H)

for h in range(n_heads):
    assert torch.equal(merged[0, :, h * d:(h + 1) * d], context[0, h])
print(f"merged[:, :, h*{d}:(h+1)*{d}] == context[:, h] for all {n_heads} heads (exact inverse of the split)")

## Cell 18 — Output projection

**What.** Apply the attention output projection `Wo` `[H, H]` to the merged heads:

```text
attn_output = merged @ Woᵀ          [1, 14, 1024]
```

**Why does multi-head attention need a final projection?**

- After merging, each head's result sits in its own fixed 64-feature block. Nothing has yet **combined** what different heads found. `Wo` mixes all 1,024 features, so every output feature can draw on every head.
- The output is about to be **added to the residual stream** (Cell 22). `Wo` maps the heads' concatenated subspaces back into the 1,024-dim coordinate system that the rest of the network reads and writes.

**Math.** `attn_output[b, s] = Wo · merged[b, s]`, a `[1024 × 1024]` matrix-vector product per token, with no bias.

**Shapes.** `merged` `[1, 14, 1024]` @ `Woᵀ` `[1024, 1024]` → `attn_output` `[1, 14, 1024]`.

**Validation.** This is the complete output of layer 0's attention sub-layer (Cells 11–18 combined). We compare it with the reference attention module's output from Cell 15.

**Expect.** The difference from the reference is at float32 rounding level (about 1e-6).

In [ ]:
# Cell 18 — Output projection: [B, S, H] @ Wo^T -> [B, S, H]

W_o = weights["layers.0.attn.Wo.weight"]                           # [H, H]
attn_output = merged @ W_o.T                                       # [1, S, 1024]

print("merged:", tuple(merged.shape), "@ Wo^T:", tuple(W_o.T.shape), "->", tuple(attn_output.shape))
assert attn_output.shape == (B, S, H)

diff = (attn_output - ref_attn_output).abs().max().item()
print(f"\nlayer-0 attention output (Cells 11-18) vs reference: max |diff| = {diff:.1e}")
assert diff < 1e-4

---
# Phase 6 — ModernBERT attention behavior

## Cell 19 — Local / sliding-window attention

**What.** Restrict each token to a **window** of nearby tokens: token `i` may attend to token `j` only if `|i − j| ≤ w`.

**Why.** Full attention compares every token with every other token, which costs `S²` scores per head per layer. For ModernBERT's 8,192-token context, that's 67 million scores per head per layer. Most useful context for a word is nearby, so **local layers** look only at `±w` neighbours. ModernBERT uses `local_attention_window = 128`, so `w = 64`.

**Math.**

```text
local:  allowed[i, j] = |i − j| ≤ w
cost:   full ≈ S²       local ≈ S · (2w + 1)        (linear in S for a fixed w)
```

Local attention is still **bidirectional**: a token sees `w` tokens on its left **and** `w` on its right. "Local" restricts distance, not direction.

**Shapes.** The window mask is boolean `[S, S]`. For readability we draw it with a toy `w = 2` on our 14 tokens. The real `w = 64` would cover this whole short sentence.

**Expect.**
- A diagonal band of width `2w + 1` in the toy mask.
- In the cost table, the local share of the full cost shrinks as `S` grows: under 2% at `S = 8,192`.

In [ ]:
# Cell 19 — Sliding-window mask: token i may attend to token j only if |i - j| <= w

def sliding_window_mask(seq_len, half_window, device=None):
    pos = torch.arange(seq_len, device=device)
    return (pos[:, None] - pos[None, :]).abs() <= half_window       # [S, S] bool


toy_w = 2
toy_mask = sliding_window_mask(S, toy_w)
print(f"Toy local mask, w = {toy_w} (✓ = may attend):")
short = [t.replace("Ġ", "_")[:6] for t in tokens_1]
print(" " * 8 + "".join(f"{t:>7}" for t in short))
for r in range(S):
    print(f"{short[r]:>8}" + "".join(f"{'✓' if toy_mask[r, c] else '·':>7}" for c in range(S)))
assert toy_mask.sum(dim=1).max() == 2 * toy_w + 1

w_real = config.local_attention_window // 2
print(f"\nReal ModernBERT local window: w = local_attention_window // 2 = {w_real} tokens each side")
print(f"\n{'S':>6} | {'full pairs S^2':>15} | {'local pairs':>12} | {'local / full':>12}")
for seq_len in [128, 512, 2048, 8192]:
    full_pairs = seq_len * seq_len
    local_pairs = int(sliding_window_mask(seq_len, w_real).sum()) if seq_len <= 2048 else seq_len * (2 * w_real + 1) - w_real * (w_real + 1)
    print(f"{seq_len:>6} | {full_pairs:>15,} | {local_pairs:>12,} | {local_pairs / full_pairs:>11.1%}")

## Cell 20 — Global attention

**What.** ModernBERT doesn't use local attention everywhere. It **alternates**: every 3rd layer (0, 3, 6, …, 27) is a **global** layer with full attention, and the other 18 layers are **local** sliding-window layers.

**Why.** Local layers are cheap but short-sighted. Information from token 0 can travel at most 64 positions per local layer. Global layers let any token reach any other token in a single step. Mixing them keeps most of the cost savings while preserving whole-document understanding. The two layer types also use different RoPE bases: global layers use `θ = 160,000` for long distances, and local layers use `θ = 10,000` for the ±64 window.

**Three ideas that are easy to conflate:**

| Concept | What it restricts | In ModernBERT |
|---|---|---|
| **Fully bidirectional** | nothing about *direction*: no causal mask, left and right context both visible | **all** 28 layers |
| **Local / sliding-window** | *distance*: only keys with `|i − j| ≤ 64` | layers 1, 2, 4, 5, …, 26 (18 layers) |
| **Global / full** | nothing: every key in the sequence | layers 0, 3, 6, 9, 12, 15, 18, 21, 24, 27 (10 layers) |

So ModernBERT isn't simply "BERT with a fixed local window". It's a bidirectional encoder whose attention **range** alternates between local and global by layer.

**Math.** Total attention pairs over all layers for a sequence of length `S`:

```text
all-global:   28 · S²
ModernBERT:   10 · S²  +  18 · S · (2·64 + 1)
```

**Expect.**
- A per-layer schedule table.
- For `S = 8,192`, ModernBERT computes about 37% of the attention pairs an all-global model would.
- An illustration of how far information can travel through local-only layers versus with a global layer.

In [ ]:
# Cell 20 — ModernBERT's layer schedule: global (full) vs local (sliding-window) attention

print(f"{'layer':>5} | {'attention':>9} | {'range':>12} | {'RoPE theta':>10}")
print("-" * 46)
for i in range(config.num_layers):
    g = config.is_global_layer(i)
    print(f"{i:>5} | {'global' if g else 'local':>9} | {'all tokens' if g else f'±{w_real} tokens':>12} | "
          f"{config.global_rope_theta if g else config.local_rope_theta:>10,.0f}")

n_global = sum(config.is_global_layer(i) for i in range(config.num_layers))
n_local = config.num_layers - n_global
assert n_global == len(range(0, config.num_layers, config.global_attn_every_n_layers))

seq_len = 8192
all_global = config.num_layers * seq_len ** 2
local_pairs = seq_len * (2 * w_real + 1) - w_real * (w_real + 1)
modernbert = n_global * seq_len ** 2 + n_local * local_pairs
print(f"\nAttention pairs for S = {seq_len:,} over {config.num_layers} layers:")
print(f"  all-global : {all_global:,}")
print(f"  ModernBERT : {modernbert:,}  ({modernbert / all_global:.1%} of all-global)")

print("\nHow far can information from token 0 travel (S = 8,192)?")
print(f"  {'after layer':>11} | {'local-only stack':>16} | {'ModernBERT schedule':>19}")
reach_local, reach_mb = 0, 0
for i in range(6):
    reach_local = min(reach_local + w_real, seq_len - 1)
    reach_mb = seq_len - 1 if config.is_global_layer(i) else min(reach_mb + w_real, seq_len - 1)
    print(f"  {i:>11} | {reach_local:>16,} | {reach_mb:>19,}")

## Cell 21 — Compare full vs local attention

**What.** Package Cells 11–18 into one attention function. Run it with a **full** mask and with a **local** mask, and show which tokens can attend to which. Then check our padding-plus-window mask against the reference implementation on a long, padded batch.

**Why.** In a real batch, the attention mask has to handle two things at once:

1. **padding:** no token may attend to a `[PAD]` key (the `attention_mask` from Cell 4)
2. **window:** in local layers, no token may attend beyond `±w`

Both become an **additive bias**: `0` where attention is allowed and a huge negative number where it is blocked, added to the scores before softmax.

**Why not literally `−∞`?** In a local layer, a padded query near the end of a short sequence can have a window containing **only** `[PAD]` keys. With `−∞` everywhere in that row, softmax computes `0/0 = NaN`. That NaN then poisons real tokens in the next layer, because `0 · NaN = NaN` inside `probs @ V`. Using the most negative finite float32 (about `−3.4e38`), as the reference implementation does, gives blocked keys an effective weight of 0 without ever producing NaN.

**Speed.** From here on, the reusable functions use `F.linear(x, W)`, which computes exactly `x @ Wᵀ` but is about 3× faster on this CPU because it avoids a transposed view. The teaching cells above keep the explicit `@`.

**Math.**

```text
allowed[b, i, j] = attention_mask[b, j] = 1   and   (global layer  or  |i − j| ≤ w)
scores'[b, h, i, j] = scores[b, h, i, j] + (0 if allowed else −3.4e38)
```

**Shapes.** The bias is `[B, 1, S, S]` and broadcasts over the heads. On our 14-token sentence we use a toy `w = 3`, because the real `w = 64` would allow everything. For the validation we use a padded batch of two sequences, the longer one over 128 tokens, with the real `w = 64`.

**Expect.**
- Under the toy window, the second `Ġcard` (position 11) can no longer see `Ġcard`, `Ġpayment` and `Ġfailed` at positions 2–4, so its output changes.
- Our boolean mask matches the reference's sliding-window mask exactly, padding included.
- Layer 1, a local layer, produces the same attention output as the reference on the long batch.

In [ ]:
# Cell 21 — One attention function (Cells 11-18) with a padding + window bias

import torch.nn.functional as F

NEG_INF = torch.finfo(torch.float32).min   # "minus infinity" that stays finite (see the note above)

def attention_bias(attention_mask, half_window=None):
    """attention_mask [B, S] (1 = real token) -> additive bias [B, 1, S, S] and boolean allowed mask."""
    B_, S_ = attention_mask.shape
    allowed = attention_mask[:, None, None, :].bool().expand(B_, 1, S_, S_)    # keys must be real tokens
    if half_window is not None:
        allowed = allowed & sliding_window_mask(S_, half_window, attention_mask.device)   # and within the window
    bias = torch.zeros(B_, 1, S_, S_, device=attention_mask.device).masked_fill(~allowed, NEG_INF)
    return bias, allowed


def self_attention(x_in, layer_idx, bias, cos, sin):
    """Cells 11-18: QKV projection, head split, RoPE, scores, softmax, weighted values, merge, Wo."""
    B_, S_, _ = x_in.shape
    # F.linear(x, W) computes exactly x @ W.T, but faster on CPU
    q, k, v = F.linear(x_in, weights[f"layers.{layer_idx}.attn.Wqkv.weight"]).chunk(3, dim=-1)  # [B, S, H] each
    Qh, Kh, Vh = split_heads(q), split_heads(k), split_heads(v)                            # [B, heads, S, d]
    Qh, Kh = apply_rope(Qh, cos, sin), apply_rope(Kh, cos, sin)
    probs = torch.softmax(Qh @ Kh.transpose(-2, -1) / d ** 0.5 + bias, dim=-1)             # [B, heads, S, S]
    out = (probs @ Vh).transpose(1, 2).reshape(B_, S_, H)                                   # [B, S, H]
    return F.linear(out, weights[f"layers.{layer_idx}.attn.Wo.weight"]), probs


mask_1 = torch.ones(1, S, dtype=torch.long)
bias_full, allowed_full = attention_bias(mask_1)
bias_toy, allowed_toy = attention_bias(mask_1, half_window=3)

out_full, probs_full = self_attention(x, 0, bias_full, cos_0, sin_0)
out_toy, probs_toy = self_attention(x, 0, bias_toy, cos_0, sin_0)
assert torch.allclose(out_full, attn_output, atol=1e-5), "full-mask path reproduces Cell 18"

i = 11
print(f"Query {tokens_1[i]!r} (position {i}) may attend to:")
print("  full      :", [tokens_1[j] for j in range(S) if allowed_full[0, 0, i, j]])
print("  local w=3 :", [tokens_1[j] for j in range(S) if allowed_toy[0, 0, i, j]])
print(f"  attention on positions 2-4 — full: {probs_full[0, 0, i, 2:5].sum():.3f} | local: {probs_toy[0, 0, i, 2:5].sum():.3f}")
print(f"  output change for this token: max |diff| = {(out_full[0, i] - out_toy[0, i]).abs().max():.3f}")

# Validation on a long, padded batch with the real window (w = 64)
from transformers.masking_utils import create_bidirectional_sliding_window_mask

long_texts = [" ".join(["My card payment failed, so I need a new card and a refund for the transfer."] * 9),
              "Short second sequence about investing in a mutual fund."]
long_batch = tokenizer(long_texts, padding=True, return_tensors="pt")
ids_long, mask_long = long_batch["input_ids"], long_batch["attention_mask"]
x_long = layer_norm(W_emb[ids_long], weights["embeddings.norm.weight"])
S_long = ids_long.shape[1]
print(f"\nlong batch: input_ids {tuple(ids_long.shape)}, real tokens per sequence {mask_long.sum(dim=1).tolist()}")

bias_local, allowed_local = attention_bias(mask_long, half_window=w_real)
ref_mask = create_bidirectional_sliding_window_mask(config=reference.config, inputs_embeds=x_long,
                                                    attention_mask=mask_long, allow_is_bidirectional_skip=False)
ref_allowed = ref_mask if ref_mask.dtype == torch.bool else (ref_mask == 0)
assert torch.equal(allowed_local, ref_allowed.expand_as(allowed_local)), "window + padding mask matches reference"
print("our window + padding mask == reference sliding-window mask")

cos_l, sin_l = rope_cos_sin(S_long, inv_freq_local)
ours_l1, _ = self_attention(layer_norm(x_long, weights["layers.1.attn_norm.weight"]), 1, bias_local, cos_l, sin_l)
cos_lr, sin_lr = reference.rotary_emb(x_long, torch.arange(S_long)[None], "sliding_attention")
with torch.no_grad():
    ref_l1, _ = reference.layers[1].attn(reference.layers[1].attn_norm(x_long), position_embeddings=(cos_lr, sin_lr), attention_mask=ref_mask)
real = mask_long.bool()
diff = (ours_l1[real] - ref_l1[real]).abs().max().item()
print(f"layer 1 (local) attention on the long padded batch vs reference: max |diff| = {diff:.1e}")
assert diff < 1e-4

---
# Phase 7 — Transformer block

## Cell 22 — Residual connection

**What.** Add the attention output back onto the layer's input:

```text
h = x + attention_output
```

**Why.** The attention sub-layer computes an **update**, not a replacement. Adding it to the input (the "residual stream") has two big benefits:

- **Information is preserved.** Whatever a layer doesn't change passes through untouched, so the original token identity survives 28 layers.
- **Optimization is easier.** The gradient of `x + f(x)` with respect to `x` is `I + ∂f/∂x`. The identity term gives gradients a direct path from the loss back to early layers, so deep stacks train stably instead of vanishing or exploding.

**Math.** `h[b, s] = x[b, s] + attn_output[b, s]`, an element-wise add of two `[1024]` vectors per token.

**Shapes.** `x` `[1, 14, 1024]` + `attn_output` `[1, 14, 1024]` → `h` `[1, 14, 1024]`.

**Expect.** The update is a fraction of the input's size: attention **nudges** each token's vector rather than overwriting it.

In [ ]:
# Cell 22 — Residual connection: h = x + attention_output

h_attn = x + attn_output                                          # [1, S, 1024]

print("x:", tuple(x.shape), "+ attention_output:", tuple(attn_output.shape), "->", tuple(h_attn.shape))
assert h_attn.shape == x.shape

print(f"\n{'token':12s} | {'|x|':>7} | {'|update|':>8} | {'|x + update|':>12} | {'update / x':>10}")
for s in [0, 2, 3, 11, S - 1]:
    nx, nu, nh = x[0, s].norm(), attn_output[0, s].norm(), h_attn[0, s].norm()
    print(f"{tokens_1[s]:12s} | {nx:7.2f} | {nu:8.2f} | {nh:12.2f} | {nu / nx:10.2f}")

## Cell 23 — Normalization

**What.** Implement the LayerNorm used throughout ModernBERT, and apply it to the post-attention residual stream (`mlp_norm`).

**Why.** As updates are added layer after layer, the scale of the residual stream drifts. Normalizing each token's vector before a sub-layer keeps that sub-layer's input at a predictable scale, which stabilizes both the forward pass and training. ModernBERT uses **pre-norm**: it normalizes the *input* to each sub-layer, while the residual stream itself stays un-normalized. LayerNorm appears in four places:

| Where | Weight |
|---|---|
| after the token embeddings | `embeddings.norm` (Cell 11) |
| before attention, layers 1–27 | `layers.{i}.attn_norm` (layer 0 has none, because the embedding norm already precedes it) |
| before the MLP, every layer | `layers.{i}.mlp_norm` |
| after the last layer | `final_norm` |

**Math.** For one token vector `v ∈ ℝ^1024`:

```text
μ = mean(v)      σ² = mean((v − μ)²)
LayerNorm(v) = (v − μ) / √(σ² + ε) · γ        ε = 1e-5,  γ = learned scale [1024]
```

ModernBERT's LayerNorm has **no bias** `β`. Each token is normalized **independently** across its 1,024 features, never across tokens or the batch.

**Shapes.** `[1, 14, 1024]` → `[1, 14, 1024]`. The `μ` and `σ` statistics are `[1, 14, 1]`.

**Expect.**
- Before: each token has an arbitrary mean and standard deviation.
- After normalizing (before the scale `γ`), every token has mean ≈ 0 and std ≈ 1.
- Our `mlp_norm` output matches the reference.

In [ ]:
# Cell 23 — LayerNorm (no bias): normalize each token across its 1,024 features, then scale by gamma

gamma = weights["layers.0.mlp_norm.weight"]                       # [1024]

mu = h_attn.mean(dim=-1, keepdim=True)                            # [1, S, 1]
var = h_attn.var(dim=-1, keepdim=True, unbiased=False)            # [1, S, 1]
normalized = (h_attn - mu) / torch.sqrt(var + config.norm_eps)    # mean 0, std 1 per token
mlp_input = normalized * gamma                                    # learned per-feature scale

print("h:", tuple(h_attn.shape), "| mu, sigma:", tuple(mu.shape), "| LayerNorm(h):", tuple(mlp_input.shape))
print(f"\n{'token':12s} | {'mean before':>11} | {'std before':>10} | {'mean after':>10} | {'std after':>9}")
for s in [0, 2, 11, S - 1]:
    print(f"{tokens_1[s]:12s} | {h_attn[0, s].mean():11.4f} | {h_attn[0, s].std(unbiased=False):10.4f} | "
          f"{normalized[0, s].mean():10.4f} | {normalized[0, s].std(unbiased=False):9.4f}")

assert torch.allclose(normalized.mean(dim=-1), torch.zeros(1, S), atol=1e-5)
assert torch.allclose(normalized.std(dim=-1, unbiased=False), torch.ones(1, S), atol=1e-3)
assert torch.allclose(mlp_input, layer_norm(h_attn, gamma), atol=1e-6), "same as the layer_norm helper from Cell 11"

with torch.no_grad():
    ref_mlp_input = reference.layers[0].mlp_norm(h_attn)
print(f"\nours vs reference mlp_norm: max |diff| = {(mlp_input - ref_mlp_input).abs().max():.1e}")
assert torch.allclose(mlp_input, ref_mlp_input, atol=1e-5)

## Cell 24 — MLP

**What.** The feed-forward network that each token passes through after attention. Start with the plain idea, `H → intermediate → H`, then implement ModernBERT's actual **gated (GeGLU)** version.

**Why.** Attention moves information **between** tokens. The MLP transforms each token **on its own**: it's where most of the per-token computation, and much of the stored knowledge, lives. ModernBERT uses a *gated* MLP. One projection produces content, a second produces a gate, and their element-wise product lets the network switch features on and off per token. Gated MLPs consistently train better than the plain version.

**Math.**

```text
plain MLP (classic BERT):      out = W_down · GELU(W_up · v)                    H → I → H

ModernBERT GeGLU:
    [a | g] = Wi · v                      Wi: [2I, H]  (content a and gate g, each [I])
    out     = Wo · (GELU(a) ⊙ g)          Wo: [H, I]
```

`GELU(z) = z · Φ(z)`, a smooth ReLU. `⊙` is element-wise multiplication. With `H = 1,024` and `I = 2,624`, `Wi` is `[5248, 1024]` and `Wo` is `[1024, 2624]`.

**Shapes.**

```text
mlp_input            [1, 14, 1024]
Wi                → [1, 14, 5248]
split             → a [1, 14, 2624],  g [1, 14, 2624]
GELU(a) ⊙ g       → [1, 14, 2624]
Wo                → mlp_output [1, 14, 1024]
```

**Expect.**
- The shapes above.
- The plain, un-gated path gives a very different output, which shows the gate matters.
- Our gated MLP matches the reference MLP.

In [ ]:
# Cell 24 — MLP: plain H -> I -> H, then ModernBERT's gated GeGLU version

import torch.nn.functional as F

W_i = weights["layers.0.mlp.Wi.weight"]                           # [2I, H] = [5248, 1024]
W_mlp_o = weights["layers.0.mlp.Wo.weight"]                       # [H, I]  = [1024, 2624]
I_ = config.intermediate_size

up = mlp_input @ W_i.T                                            # [1, S, 2I]
a, g = up.chunk(2, dim=-1)                                        # content a, gate g: [1, S, I] each

# Plain MLP idea (not what ModernBERT computes): H -> I -> H with only the content half
plain_output = F.gelu(a) @ W_mlp_o.T                              # [1, S, H]

# ModernBERT's gated MLP (GeGLU)
gated = F.gelu(a) * g                                             # [1, S, I]
mlp_output = gated @ W_mlp_o.T                                    # [1, S, H]

print("mlp_input:", tuple(mlp_input.shape), "-> Wi:", tuple(up.shape), "-> a, g:", tuple(a.shape), tuple(g.shape))
print("GELU(a) * g:", tuple(gated.shape), "-> Wo -> mlp_output:", tuple(mlp_output.shape))
assert up.shape[-1] == 2 * I_ and gated.shape[-1] == I_ and mlp_output.shape == mlp_input.shape

frac_off = (gated.abs() < 1e-3).float().mean().item()
print(f"\nshare of intermediate features the gate effectively switches off (|GELU(a)·g| < 1e-3): {frac_off:.1%}")
print(f"plain vs gated output: max |diff| = {(plain_output - mlp_output).abs().max():.2f}  (the gate changes the result)")

with torch.no_grad():
    ref_mlp_output = reference.layers[0].mlp(mlp_input)
diff = (mlp_output - ref_mlp_output).abs().max().item()
print(f"ours vs reference MLP: max |diff| = {diff:.1e}")
assert diff < 1e-4

## Cell 25 — Complete Transformer block

**What.** Combine everything into one encoder layer and print every tensor shape along the way:

```text
input h
 ↓  normalization (attn_norm; none in layer 0)
 ↓  attention (global or local mask, RoPE with the matching θ)
 ↓  residual:  h = h + attention
 ↓  normalization (mlp_norm)
 ↓  MLP (GeGLU)
 ↓  residual:  h = h + MLP
output h
```

**Why.** This pre-norm block is the unit that repeats 28 times. Getting one block exactly right, and verifying it against the reference, is what makes the whole stack trustworthy.

**Math.**

```text
h' = h + Attn(norm_attn(h))         (norm_attn = identity for layer 0)
h''= h' + MLP(norm_mlp(h'))
```

**Shapes.** Every intermediate is `[B, S, 1024]` except inside attention (`[B, 16, S, 64]`, `[B, 16, S, S]`) and inside the MLP (`[B, S, 5248]` → `[B, S, 2624]`).

**Expect.** The printed shape trace for layer 0, and an exact-to-rounding match with the reference implementation's full layer-0 output.

In [ ]:
# Cell 25 — One complete pre-norm ModernBERT encoder layer

def mlp(m_in, layer_idx):
    a_, g_ = F.linear(m_in, weights[f"layers.{layer_idx}.mlp.Wi.weight"]).chunk(2, dim=-1)
    return F.linear(F.gelu(a_) * g_, weights[f"layers.{layer_idx}.mlp.Wo.weight"])


def encoder_layer(h, layer_idx, bias, cos, sin, verbose=False):
    show = print if verbose else (lambda *args, **kw: None)
    show(f"layer {layer_idx} ({'global' if config.is_global_layer(layer_idx) else 'local'})")
    show("  input h                 ", tuple(h.shape))
    a_in = h if layer_idx == 0 else layer_norm(h, weights[f"layers.{layer_idx}.attn_norm.weight"])
    show("  attn_norm(h)            ", tuple(a_in.shape), "(identity in layer 0)" if layer_idx == 0 else "")
    attn_out, probs = self_attention(a_in, layer_idx, bias, cos, sin)
    show("  attention probs         ", tuple(probs.shape))
    show("  attention output        ", tuple(attn_out.shape))
    h = h + attn_out
    show("  h + attention           ", tuple(h.shape))
    m_in = layer_norm(h, weights[f"layers.{layer_idx}.mlp_norm.weight"])
    show("  mlp_norm(h)             ", tuple(m_in.shape))
    m_out = mlp(m_in, layer_idx)
    show("  MLP output              ", tuple(m_out.shape))
    h = h + m_out
    show("  h + MLP  (layer output) ", tuple(h.shape))
    return h


h0 = encoder_layer(x, 0, bias_full, cos_0, sin_0, verbose=True)

with torch.no_grad():
    ref_h0 = reference.layers[0](x_ref, attention_mask=None, position_embeddings=(cos_ref0, sin_ref0))
diff = (h0 - ref_h0).abs().max().item()
print(f"\nours vs reference full layer 0: max |diff| = {diff:.1e}")
assert diff < 1e-4

---
# Phase 8 — Encoder stack

## Cell 26 — One encoder layer

**What.** Pass a real **padded batch** through one global layer (layer 0) and then one local layer (layer 1), using the encoder layer from Cell 25 with the right mask and RoPE for each layer type.

**Why.** A single short sentence doesn't exercise everything. This batch has two sequences: a long one (over 128 tokens, so the ±64 window really cuts off distant tokens) and a short one padded with `[PAD]`. Matching the reference here confirms that padding masks, sliding windows and both RoPE bases are all wired correctly in a complete layer.

**Math.**

```text
h1 = Layer_0(x_long; global bias, RoPE θ = 160k)
h2 = Layer_1(h1;     local bias (±64), RoPE θ = 10k)
```

**Shapes.** `x_long` `[2, S_long, 1024]` → `h1` `[2, S_long, 1024]` → `h2` `[2, S_long, 1024]`. The `[PAD]` positions are excluded from the comparison, because their outputs are never used.

**Expect.** Both layers match the reference on every real token, to float32 rounding.

In [ ]:
# Cell 26 — A padded batch through one global layer (0) and one local layer (1)

bias_global_long, _ = attention_bias(mask_long)
bias_local_long, _ = attention_bias(mask_long, half_window=w_real)
cos_g_long, sin_g_long = rope_cos_sin(S_long, inv_freq_global)
cos_l_long, sin_l_long = rope_cos_sin(S_long, inv_freq_local)

h1 = encoder_layer(x_long, 0, bias_global_long, cos_g_long, sin_g_long)
h2 = encoder_layer(h1, 1, bias_local_long, cos_l_long, sin_l_long)
print("x_long:", tuple(x_long.shape), "-> layer 0 (global):", tuple(h1.shape), "-> layer 1 (local):", tuple(h2.shape))

from transformers.masking_utils import create_bidirectional_mask
ref_full_mask = create_bidirectional_mask(config=reference.config, inputs_embeds=x_long, attention_mask=mask_long,
                                          allow_is_bidirectional_skip=False)
cos_gr, sin_gr = reference.rotary_emb(x_long, torch.arange(S_long)[None], "full_attention")
with torch.no_grad():
    ref_h1 = reference.layers[0](x_long, attention_mask=ref_full_mask, position_embeddings=(cos_gr, sin_gr))
    ref_h2 = reference.layers[1](ref_h1, attention_mask=ref_mask, position_embeddings=(cos_lr, sin_lr))

d1 = (h1[real] - ref_h1[real]).abs().max().item()
d2 = (h2[real] - ref_h2[real]).abs().max().item()
print(f"\nlayer 0 vs reference (real tokens): max |diff| = {d1:.1e}")
print(f"layer 1 vs reference (real tokens): max |diff| = {d2:.1e}")
assert d1 < 1e-4 and d2 < 1e-4

## Cell 27 — Stack layers

**What.** Build all 28 layers as an `nn.ModuleList`, add the embedding and final normalization, and run the **complete ModernBERT encoder** end to end. Then compare its output with the reference model's `last_hidden_state`.

**Why.** This is our ModernBERT: real pretrained weights, with every computation done by the code in this notebook. Each `ModernBertLayer` module just holds its layer's pretrained tensors as (frozen) parameters and calls `encoder_layer` from Cell 25. Wrapping them in `nn.ModuleList` makes the 28 layers an ordinary PyTorch model we can iterate over, freeze or fine-tune (Phase 12).

**Math.**

```text
h₀ = LayerNorm(W_emb[input_ids])
h_{i+1} = Layer_i(h_i)    with global bias / θ = 160k if i % 3 == 0, else local bias / θ = 10k
output = LayerNorm_final(h₂₈)
```

**Shapes.** `input_ids` `[B, S]` → `h₀` `[B, S, 1024]` → (28 layers) → `output` `[B, S, 1024]`.

**Expect.**
- 28 layers, 10 global and 18 local.
- 394,781,696 parameters in total, all frozen.
- On the long padded batch, our output matches the reference on every real token, to about 3e-5 after 28 layers of float32 arithmetic (the cell asserts < 1e-3).
- The timing shows how long one forward pass takes on this CPU.

In [ ]:
# Cell 27 — The full ModernBERT encoder: nn.ModuleList of L layers (28 for large) + embeddings + final norm

import time

import torch.nn as nn


class ModernBertLayer(nn.Module):
    """Holds one layer's pretrained tensors (shared storage, frozen) and runs encoder_layer() from Cell 25."""

    def __init__(self, layer_idx):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_global = config.is_global_layer(layer_idx)
        prefix = f"layers.{layer_idx}."
        self.params = nn.ParameterDict({
            name[len(prefix):].replace(".", "_"): nn.Parameter(t, requires_grad=False)
            for name, t in weights.items() if name.startswith(prefix)
        })

    def forward(self, h, bias, cos, sin):
        return encoder_layer(h, self.layer_idx, bias, cos, sin)


layers = nn.ModuleList([ModernBertLayer(i) for i in range(config.num_layers)])


def modernbert_forward(input_ids, attention_mask, return_all_layers=False):
    """Our ModernBERT encoder: [B, S] token IDs -> [B, S, H] contextual representations."""
    S_ = input_ids.shape[1]
    h = layer_norm(W_emb[input_ids], weights["embeddings.norm.weight"])
    masks = {True: attention_bias(attention_mask)[0],
             False: attention_bias(attention_mask, half_window=config.local_attention_window // 2)[0]}
    ropes = {True: rope_cos_sin(S_, inv_freq_global), False: rope_cos_sin(S_, inv_freq_local)}
    all_layers = [h]
    for layer in layers:
        h = layer(h, masks[layer.is_global], *ropes[layer.is_global])
        all_layers.append(h)
    output = layer_norm(h, weights["final_norm.weight"])
    return (output, all_layers) if return_all_layers else output


n_params = sum(p.numel() for p in layers.parameters()) + W_emb.numel() + 2 * H
print(f"layers: {len(layers)} ({sum(l.is_global for l in layers)} global, {sum(not l.is_global for l in layers)} local)")
print(f"parameters (layers + embeddings + 2 norms): {n_params:,} | trainable: {sum(p.numel() for p in layers.parameters() if p.requires_grad)}")
assert n_params == sum(t.numel() for t in weights.values())

with torch.no_grad():
    t0 = time.perf_counter()
    ours = modernbert_forward(ids_long, mask_long)
    t_ours = time.perf_counter() - t0
    t0 = time.perf_counter()
    ref_out = reference(input_ids=ids_long, attention_mask=mask_long).last_hidden_state
    t_ref = time.perf_counter() - t0

diff = (ours[real] - ref_out[real]).abs().max().item()
print(f"\ninput_ids {tuple(ids_long.shape)} -> our output {tuple(ours.shape)}")
print(f"ours vs reference last_hidden_state (real tokens): max |diff| = {diff:.1e}")
print(f"forward-pass time on CPU: ours {t_ours:.2f}s | reference {t_ref:.2f}s")
assert diff < 1e-3
encoder_validation_diff = diff

## Cell 28 — Inspect representations layer by layer

**What.** Capture the hidden state after every layer for `My card payment failed, so I need a new card.` and watch how token representations change with depth.

**Why.** The sentence contains the token `Ġcard` **twice**: once as the card whose payment failed, and once as the new card being requested. At the embedding layer, both are the *same row of the table*, so they're identical. Every layer mixes in context, so the two occurrences should drift apart as the model builds context-specific meanings.

**Math.** For each layer `ℓ` (0 = embedding output, 28 = last layer) we measure:

- `cos(h_ℓ[card₁], h_ℓ[card₂])`: how similar the two `card` tokens are
- `cos(h_ℓ[s], h_{ℓ−1}[s])`, averaged over tokens: how much a layer changes the representations

**Shapes.** 29 hidden states (embeddings plus 28 layers), each `[1, 14, 1024]`.

**Expect.**
- The two `card` vectors start with cosine **1.0** at the embedding layer. They drift apart through the middle layers (lowest about 0.88, around layer 14), then partly re-converge (about 0.97 near layers 24–26, 0.94 at the last layer). Token identity stays the dominant component of each vector, and context changes its direction by a few percent. That's enough to separate the two meanings, because downstream layers and heads read exactly these directions.
- Layer 0 changes the representations the most (layer-to-layer cosine of about 0.59), because it turns raw embeddings into contextual vectors. After that, each layer refines rather than rewrites (cosine about 0.92–0.99), which is the residual stream at work.

In [ ]:
# Cell 28 — Representations after every layer

with torch.no_grad():
    _, all_hidden = modernbert_forward(input_ids_1, torch.ones_like(input_ids_1), return_all_layers=True)

print(f"captured {len(all_hidden)} hidden states (embeddings + {config.num_layers} layers), each {tuple(all_hidden[0].shape)}")
c1, c2 = 2, 11
assert tokens_1[c1] == tokens_1[c2] == "Ġcard"

cos = torch.nn.functional.cosine_similarity
card_sim = [cos(hs[0, c1], hs[0, c2], dim=0).item() for hs in all_hidden]
step_sim = [float("nan")] + [cos(all_hidden[l][0], all_hidden[l - 1][0], dim=-1).mean().item() for l in range(1, len(all_hidden))]

print(f"\n{'layer':>5} | {'type':>6} | {'cos(card@2, card@11)':>20} | {'mean cos to previous layer':>26}")
for l in range(len(all_hidden)):
    kind = "embed" if l == 0 else ("global" if config.is_global_layer(l - 1) else "local")
    print(f"{l:>5} | {kind:>6} | {card_sim[l]:>20.3f} | {step_sim[l]:>26.3f}")
assert abs(card_sim[0] - 1.0) < 1e-5, "identical at the embedding layer"
assert min(card_sim[1:]) < 0.95, "context separates the two occurrences"
print(f"\nlowest card similarity: {min(card_sim):.3f} at layer {card_sim.index(min(card_sim))}; final layer: {card_sim[-1]:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(len(all_hidden)), card_sim, marker="o", label="cos(card@2, card@11)")
ax.plot(range(len(all_hidden)), step_sim, marker=".", label="mean cos to previous layer")
ax.set_xlabel("layer (0 = embedding output)")
ax.set_ylabel("cosine similarity")
ax.set_title("The two 'card' tokens diverge as context is mixed in")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

---
# Phase 9 — Understand the output

## Cell 29 — Encoder output

**What.** Make the key point explicit: ModernBERT's output is

```text
[B, S, H]     one contextual 1,024-dim vector for every input token
```

It does **not** inherently produce a next token. Unlike SmolLM2, ModernBERT has no `[B, S, V]` next-token logits at the end of its stack.

**Why.** A decoder's last layer feeds an LM head that scores the *next* token. An encoder's output is a set of representations, and what you do with them is up to the **head** you put on top. The pretrained checkpoint ships with one such head, the **masked-LM head** used in pretraining. It fills in a `[MASK]` from **both** sides, which is a different task from continuing text. We use that head once here to show what the representations already contain. For routing, we put our own compatibility head on top instead (Phase 11).

**Math.** MLM head, applied only at the `[MASK]` position:

```text
z = LayerNorm(GELU(W_dense · h_mask))           W_dense [1024, 1024]
logits = W_emb · z + b                          tied to the token-embedding table, [V] = 50,368 scores
```

**Shapes.** Encoder output `[1, S, 1024]`. At the mask position: `h_mask` `[1024]` → `logits` `[50368]`.

**Expect.** The encoder output is `[1, S, 1024]`, with no vocabulary dimension. For `My card payment [MASK], so I need a new card.`, the top predictions are plausible words such as `expired`, `failed`, `bounced` or `declined`, chosen using context on **both** sides of the blank.

In [ ]:
# Cell 29 — Encoder output [B, S, H]; a masked-LM demo with the checkpoint's pretraining head

from safetensors import safe_open

masked_text = "My card payment [MASK], so I need a new card."
mlm_batch = tokenizer(masked_text, return_tensors="pt")
mlm_ids = mlm_batch["input_ids"]
with torch.no_grad():
    encoder_out = modernbert_forward(mlm_ids, mlm_batch["attention_mask"])

print("input_ids:", tuple(mlm_ids.shape), "-> encoder output:", tuple(encoder_out.shape), "= [B, S, H]")
assert encoder_out.shape == (1, mlm_ids.shape[1], H), "one vector per token, no vocabulary dimension"

with safe_open(weights_path, "pt") as f:
    head_dense = f.get_tensor("head.dense.weight")                 # [1024, 1024]
    head_norm = f.get_tensor("head.norm.weight")                   # [1024]
    decoder_bias = f.get_tensor("decoder.bias")                    # [V]

mask_pos = (mlm_ids[0] == MASK_ID).nonzero().item()
z = layer_norm(F.gelu(encoder_out[0, mask_pos] @ head_dense.T), head_norm)    # [1024]
logits = z @ W_emb.T + decoder_bias                                            # [V] (tied embeddings)

print(f"\n[MASK] at position {mask_pos}: logits {tuple(logits.shape)}")
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, k=5)
mlm_top_prediction = tokenizer.decode([top.indices[0].item()])
print(f"Top predictions for: {masked_text}")
for p_, tid in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {tokenizer.decode([tid])!r:14s} {p_:.3f}")

## Cell 30 — Token representation

**What.** Take one token's final vector, the first `Ġcard`, and inspect it. Then show that it depends on words to its **right**.

**Why.** Each output vector is a summary of that token **in context**. Because every layer is bidirectional, a token's final vector includes information from tokens on both sides. To prove the right side matters, encode two sentences whose words **up to and including `card`** are identical and which differ only *after* it:

```text
"My card payment failed …"
"My card payment succeeded …"
```

A causal decoder would give `card` exactly the same vector in both, because it can't see the future. ModernBERT gives different vectors.

**Math.** `v = output[0, position_of_card]` ∈ ℝ^1024. We compare `v` across the two sentences with cosine similarity and the max absolute difference.

**Shapes.** One token vector is `[1024]`. Each sentence's output is `[1, S, 1024]`.

**Expect.** The vector is 1,024-dimensional, and the `card` vectors differ noticeably (cosine about 0.98) between the two sentences even though everything to the left is identical.

In [ ]:
# Cell 30 — One token's vector, and proof that it carries right-hand context

card_vec = encoder_out[0, 2]
print(f"token {tokenizer.convert_ids_to_tokens(mlm_ids[0, 2].item())!r}: vector shape {tuple(card_vec.shape)}, "
      f"norm {card_vec.norm():.2f}, first 6 dims {card_vec[:6]}")
assert card_vec.shape == (H,)

pair = ["My card payment failed and I am upset.", "My card payment succeeded and I am happy."]
pair_batch = tokenizer(pair, padding=True, return_tensors="pt")
toks = [tokenizer.convert_ids_to_tokens(r) for r in pair_batch["input_ids"]]
assert toks[0][:4] == toks[1][:4], "identical up to and including 'Ġpayment'"
with torch.no_grad():
    pair_out = modernbert_forward(pair_batch["input_ids"], pair_batch["attention_mask"])

pos = toks[0].index("Ġcard")
v_a, v_b = pair_out[0, pos], pair_out[1, pos]
print(f"\nleft context of 'Ġcard' in both sentences: {toks[0][:pos + 1]}")
print(f"cosine('Ġcard' in A, 'Ġcard' in B) = {torch.nn.functional.cosine_similarity(v_a, v_b, dim=0):.4f}")
print(f"max |difference|                    = {(v_a - v_b).abs().max():.4f}")
print("A causal decoder would give identical vectors here; ModernBERT's differ because it reads the right context.")
assert (v_a - v_b).abs().max() > 1e-2

## Cell 31 — Sequence representation

**What.** Turn per-token outputs `[B, S, H]` into **one vector per sequence**, `[B, H]`, and choose the pooling strategy for the router.

**Options.**

| Pooling | Definition | Notes |
|---|---|---|
| `[CLS]` | `output[:, 0]` | One position. BERT trained it with next-sentence prediction, but ModernBERT is pretrained **only** with masked LM, so nothing specifically trains `[CLS]` to summarize the sequence. |
| **mean** | average over **real** tokens (padding excluded via the mask) | Uses every token's contextual vector. It's the pooling ModernBERT's checkpoint itself is configured with (`classifier_pooling = "mean"`). |
| max | element-wise max over real tokens | Sensitive to single extreme features. |

**Why it matters for routing.** Our input is a whole conversation plus an agent description. A frozen encoder's `[CLS]` vector hasn't been trained to summarize that pair, whereas the mean of all contextual token vectors reflects the content of the entire input.

**Math.** Mask-aware mean, so padding never dilutes the average:

```text
pooled[b] = Σ_s mask[b, s] · output[b, s]  /  Σ_s mask[b, s]
```

**A quick check.** We encode four sentences: an anchor, a paraphrase of it, a related card request, and an unrelated investment request. A good sentence vector should rank paraphrase > related > unrelated.

**Choice.** **Mean pooling** is the most principled option for a frozen MLM-pretrained encoder, it matches the checkpoint's own configuration, and it uses the whole input. We use it for the rest of the notebook.

**Shapes.** `[4, S, 1024]` → `[4, 1024]` for each method.

**Expect.**
- All three methods rank the sentences correctly here.
- Mean pooling gives the clearest margins: paraphrase about 0.98, related about 0.90, unrelated about 0.85.
- `[CLS]` barely separates related from unrelated (about 0.96 vs 0.96), and max pooling sits in between (0.97 / 0.92 / 0.90).
- All the similarities are high (0.85 and above), which is typical of raw MLM representations. Fine distinctions live in small directions of the vector space, which is exactly what a trained head (Phase 12) learns to read.

In [ ]:
# Cell 31 — Sequence representation: [B, S, H] -> [B, H]; we choose mask-aware mean pooling

def mean_pool(hidden, attention_mask):
    m = attention_mask[..., None].to(hidden.dtype)                 # [B, S, 1]
    return (hidden * m).sum(dim=1) / m.sum(dim=1)                  # [B, H]


def cls_pool(hidden, attention_mask):
    return hidden[:, 0]                                            # [B, H]


def max_pool(hidden, attention_mask):
    return hidden.masked_fill(~attention_mask[..., None].bool(), float("-inf")).max(dim=1).values


sentences = ["My card is blocked and I can't use it.",               # anchor
             "I can't use my card because it has been blocked.",    # paraphrase
             "Please send me a replacement debit card.",            # related (card)
             "I want to start a monthly SIP in a mutual fund."]     # unrelated
batch4 = tokenizer(sentences, padding=True, return_tensors="pt")
with torch.no_grad():
    out4 = modernbert_forward(batch4["input_ids"], batch4["attention_mask"])
print("encoder output:", tuple(out4.shape), "-> pooled:", tuple(mean_pool(out4, batch4["attention_mask"]).shape))

print(f"\n{'pooling':>7} | {'paraphrase':>10} | {'related':>8} | {'unrelated':>9} | ranks paraphrase > related > unrelated?")
for name, pool in [("[CLS]", cls_pool), ("mean", mean_pool), ("max", max_pool)]:
    v = pool(out4, batch4["attention_mask"])
    sims = torch.nn.functional.cosine_similarity(v[0:1], v[1:], dim=-1)
    print(f"{name:>7} | {sims[0]:10.3f} | {sims[1]:8.3f} | {sims[2]:9.3f} | {bool(sims[0] > sims[1] > sims[2])}")

print(f"\ncheckpoint's own classifier_pooling: {raw_config['classifier_pooling']!r}")
print("Chosen for the router: mask-aware MEAN pooling.")
assert raw_config["classifier_pooling"] == "mean"

---
# Phase 10 — A classification head on the encoder

## Cell 32 — Fixed classification head

**What.** Put the most common kind of head on top of the encoder: a linear layer from the pooled vector to a **fixed number of classes**, here six:

```text
pooled  [B, 1024]  →  Linear(1024, 6)  →  logits  [B, 6]
```

**Why.** This is the key idea for using an encoder: it does the language work once, and a **small head** turns its pooled vector into a decision. The encoder's weights can stay frozen, or be fine-tuned along with the head. This is how "BERT for classification" is built, and it works well when the set of labels is **fixed**: sentiment, spam, toxicity, language, a fixed intent taxonomy. It's also the obvious first idea for routing: one output per agent. Cell 33 shows where that breaks.

**Math.** `logits = W · v + b`, with `W` `[6, 1024]` and `b` `[6]`. Output `k` is `w_k · v + b_k`: the dot product of the input with a **learned prototype row** `w_k` for class `k`. A `1024 → 6` layer therefore hard-wires exactly six outputs, one row per class. The meaning of each class lives entirely in its row of `W`, and no description of the class is ever an input.

**Shapes.** `W` `[6, 1024]`, `b` `[6]`. The four pooled sentences from Cell 31 (`[4, 1024]`) give `logits` `[4, 6]`.

**Expect.** A `[4, 6]` logit matrix. The head is untrained, so the numbers are arbitrary. The point is the shape: one column per fixed class.

In [ ]:
# Cell 32 — A fixed 6-way classification head: Linear(H -> 6)

torch.manual_seed(42)

NUM_CLASSES = 6
fixed_head = nn.Linear(H, NUM_CLASSES)

pooled4 = mean_pool(out4, batch4["attention_mask"])                # [4, 1024]
with torch.no_grad():
    fixed_logits = fixed_head(pooled4)                             # [4, 6]

print("W:", tuple(fixed_head.weight.shape), "| b:", tuple(fixed_head.bias.shape))
print("pooled:", tuple(pooled4.shape), "-> logits:", tuple(fixed_logits.shape))
assert fixed_logits.shape == (4, NUM_CLASSES)

manual = pooled4[0] @ fixed_head.weight[2] + fixed_head.bias[2]
assert torch.allclose(manual, fixed_logits[0, 2], atol=1e-5)
print("logit k = w_k · v + b_k  (checked for sentence 0, class 2)")
print("\nUntrained logits (rows = sentences, columns = the 6 fixed classes):")
print(fixed_logits)

## Cell 33 — Where a fixed head fails: agents that change

**What.** Show two consequences of the fixed head, using a conversation and two different sets of agent descriptions.

**Why.** For a multi-agent system, the set of agents and their responsibilities **change**: agents are added, merged or re-scoped. A fixed classifier can't follow such changes without retraining, because it never reads the agent descriptions at all.

**This is a limit of the fixed head, not of the idea.** If the agent set were fixed and we had many labelled conversations per agent, a fixed head could route fine; that's ordinary intent classification. What fails here is *one output per class* for classes that change. So we keep "head on top of a pooled encoder vector", and change what the head scores: the candidate goes **into the input**, and the head outputs one score per pair.

```text
fixed classifier:        input                      → one of K known classes
dynamic compatibility:   (input, candidate choice)  → score
```

| | Fixed classifier (this cell) | Dynamic compatibility (02 onward) |
|---|---|---|
| Model input | conversation only | conversation **+ candidate agent description** |
| Output | `K` logits, one per pre-defined class | **one** score per (conversation, candidate) pair |
| Agent identity | a row of `W`, learned by heart | the description text, read every time |
| Add a new agent | new output neuron + retraining | write a description, score it |
| Change an agent's scope | retraining | edit the description |

**Math.** Classifier: `p(class | conversation) = softmax(W · v)`. The class set is the index range of `W`. Compatibility: `score(c, a) = f(ModernBERT(c ⊕ a))`, where the agent `a` is **part of the input**.

| Works well for | fixed label sets: sentiment, spam, a fixed intent list | choices that change: agents, options written at run time |

**Expect.**
- Changing every agent description leaves the classifier's logits **exactly unchanged**, because it never sees them.
- Adding a seventh agent forces a **new** `Linear(H, 7)`, whose new row is untrained random noise.

In [ ]:
# Cell 33 — The fixed classifier never reads agent descriptions, and cannot grow

conversation = "user: My bank transfer failed."
descriptions_v1 = {"Payments": "Handles bank transfers and payment failures.", "Card Support": "Handles card blocking."}
descriptions_v2 = {"Payments": "Handles ONLY mortgage questions now.", "Card Support": "Handles bank transfers too."}

conv_batch = tokenizer([conversation], return_tensors="pt")
with torch.no_grad():
    conv_vec = mean_pool(modernbert_forward(conv_batch["input_ids"], conv_batch["attention_mask"]), conv_batch["attention_mask"])
    logits_v1 = fixed_head(conv_vec)          # descriptions_v1 are not an input...
    logits_v2 = fixed_head(conv_vec)          # ...and neither are descriptions_v2

print("logits with descriptions v1:", logits_v1)
print("logits with descriptions v2:", logits_v2)
assert torch.equal(logits_v1, logits_v2)
print("-> identical: rewriting what the agents do changes nothing, because the classifier never reads descriptions.")

torch.manual_seed(0)
fixed_head_7 = nn.Linear(H, 7)                # a 7th agent needs a NEW output neuron...
with torch.no_grad():
    fixed_head_7.weight[:6] = fixed_head.weight
    fixed_head_7.bias[:6] = fixed_head.bias
print(f"\nAdding a 7th agent: weight {tuple(fixed_head.weight.shape)} -> {tuple(fixed_head_7.weight.shape)}; "
      f"row 6 is random (norm {fixed_head_7.weight[6].norm():.3f}) until the whole classifier is retrained.")

---
# Where this goes next

We now have a verified ModernBERT forward pass and a clear view of what comes out of it: one contextual vector per token, and mean pooling to turn a sequence into one vector.

**The idea to carry forward:** a small head on top of the pooled encoder vector turns the encoder into a model that decides. Phase 10 tested the most common form, one output per fixed class. It's the right tool for fixed label sets, but it can't route a conversation to agents that are only known at run time: it never reads the agents' descriptions, and adding an agent means retraining.

**The idea stays; the head changes.** Whether it works then becomes an experiment, and the next two notebooks are that experiment, failures included:
- **`02_frozen_router.ipynb`** (branch `frozen-router`) keeps this encoder **frozen** and puts the agent's description into the input. A small trained head scores each (conversation, agent) pair, which makes it a cross-encoder router. It routes well, but on questions that need reading the text, it hits a ceiling: a head can only use what the frozen vectors already carry.
- **`03_lora_signpost.ipynb`** (branch `signpost`) lets the encoder learn too (LoRA) under the same kind of head. That closes the gap, and it became the published model, **Signpost**.